# Трансформеры — базовые и дополнительные эксперименты

Исходные Colab (копии оригиналов):
- [Трансформеры.ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%D0%A2%D1%80%D0%B0%D0%BD%D1%81%D1%84%D0%BE%D1%80%D0%BC%D0%B5%D1%80%D1%8B.ipynb)
- [ru_sentiment_research_FINAL.ipynb](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/ru_sentiment_research_FINAL.ipynb)

Перенос исходного кода; выполнение и метрики пока не проверены. Исторические результаты в оригиналах.

In [ ]:
from pathlib import Path
import os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'download_dataset.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Clone SpeechSense and run from its root or notebooks directory')
os.chdir(ROOT)
DATA_PATH = ROOT / 'data' / 'merged_sentiment_dataset.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python data/download_dataset.py first')


In [ ]:
!pip install imbalanced-learn optuna plotly -q
!pip install -q transformers datasets accelerate torchmetrics
!pip install -q sentencepiece  # нужен для некоторых токенизаторов
!pip install -q nlpaug  # для контекстной аугментации
!pip install -q sacremoses  # для back-translation

import os
import re
import gc
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Маппинг классов
CLASS_MAP = {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"}
CLASS_NAMES = [CLASS_MAP[i] for i in range(3)]
NUM_CLASSES = 3

print(" Все библиотеки импортированы!")

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_PATH, compression='gzip')

In [ ]:
print("ОЧИСТКА ТЕКСТА")

def clean_text(text: str) -> str:
    """Базовая очистка текста для трансформеров (минимальная!)."""
    if not isinstance(text, str):
        return ""
    text = text[:10000]  # обрезаем экстремально длинные
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[@#]\w+', ' ', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r'[^а-яёa-z0-9\s.!?,-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f"Размер ДО очистки: {df.shape[0]:,}")
df_clean = df.copy()
df_clean['text_clean'] = df_clean['text'].astype(str).apply(clean_text)

# Фильтр коротких текстов
MIN_LEN = 10
df_clean = df_clean[df_clean['text_clean'].str.len() >= MIN_LEN].copy()
print(f"Размер ПОСЛЕ очистки: {df_clean.shape[0]:,}")

# Распределение после очистки
print("\n=== Распределение классов в очищенном датасете ===")
print(df_clean['sentiment'].value_counts().sort_index())
print("\nДоли (%):")
print((df_clean['sentiment'].value_counts(normalize=True).sort_index() * 100).round(2))


In [ ]:
print("РАЗДЕЛЕНИЕ ДАННЫХ (70/15/15)")

from sklearn.model_selection import train_test_split

# Стратифицированное разделение 70/15/15
train_df, temp_df = train_test_split(
    df_clean, test_size=0.3, random_state=SEED,
    stratify=df_clean['sentiment']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED,
    stratify=temp_df['sentiment']
)

print(f" Train: {len(train_df):,} ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f" Validation: {len(val_df):,} ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f" Test: {len(test_df):,} ({len(test_df)/len(df_clean)*100:.1f}%)")

print(f"\n Распределение в train:")
print(train_df['sentiment'].value_counts().sort_index())
print(f"\n Распределение в validation:")
print(val_df['sentiment'].value_counts().sort_index())
print(f"\n Распределение в test:")
print(test_df['sentiment'].value_counts().sort_index())

# Трансформеры

In [ ]:
print("УНИВЕРСАЛЬНАЯ ФУНКЦИЯ ОБУЧЕНИЯ")

import os
from typing import Dict, List, Optional, Tuple

# =============================================================================
# 1. Кастомный Dataset для трансформеров
# =============================================================================
class SentimentDataset(Dataset):
    """
    Кастомный Dataset для трансформеров.
    Превращает текст в токены, добавляет padding и attention mask.
    """
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = [str(t) for t in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Токенизация текста
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

print(" Класс SentimentDataset определён")

# =============================================================================
# 2. Функция оценки модели
# =============================================================================
def evaluate_model(model, data_loader, device):
    """Оценка модели на validation/test."""
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )
            preds.extend(torch.argmax(out.logits, dim=-1).cpu().numpy())
            labels.extend(batch['labels'].cpu().numpy())

    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    return macro_f1, accuracy

print(" Функция evaluate_model определена")

# =============================================================================
# 3. Универсальная функция обучения трансформера
# =============================================================================
def train_transformer(
    model_name: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    text_col: str = 'text',
    label_col: str = 'sentiment',
    max_length: int = 128,
    batch_size: int = 16,
    learning_rate: float = 2e-5,
    epochs: int = 3,
    use_differential_lr: bool = False,
    label_smoothing: float = 0.0,
    gradient_clip_norm: float = 1.0,
    custom_loss_fn: Optional[nn.Module] = None,  # ← НОВЫЙ ПАРАМЕТР
    model_key: str = "",
    save_dir: Optional[str] = None,
) -> Dict:
    """
    Универсальная функция обучения трансформера.

    Args:
        model_name: название модели (например, 'DeepPavlov/rubert-base-cased')
        train_df: обучающая выборка
        val_df: валидационная выборка (НЕ тест!)
        text_col: колонка с текстом
        label_col: колонка с метками
        max_length: максимальная длина последовательности
        batch_size: размер батча
        learning_rate: скорость обучения
        epochs: количество эпох
        use_differential_lr: использовать дифференциальные LR
        label_smoothing: коэффициент label smoothing
        gradient_clip_norm: норма для gradient clipping
        custom_loss_fn: кастомная функция потерь (например, с весами классов)
        model_key: ключ модели для логирования
        save_dir: директория для сохранения модели

    Returns:
        Dict с результатами обучения
    """
    print(f"   Модель: {model_name}")
    print(f"   Train: {len(train_df)}, Val: {len(val_df)}")
    print(f"   Batch size: {batch_size}, LR: {learning_rate}, Epochs: {epochs}")
    print(f"   Max length: {max_length}")

    # 1. Загрузка токенизатора и модели
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3
    ).to(DEVICE)

    # 2. Создание DataLoader (train + val)
    train_dataset = SentimentDataset(
        train_df[text_col].tolist(),
        train_df[label_col].tolist(),
        tokenizer, max_length,
    )
    val_dataset = SentimentDataset(
        val_df[text_col].tolist(),
        val_df[label_col].tolist(),
        tokenizer, max_length,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f" Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

    # 3. Оптимизатор
    if use_differential_lr:
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped = [
            {'params': [p for n, p in model.bert.named_parameters()
                        if not any(nd in n for nd in no_decay)
                        and any(f'layer.{i}.' in n for i in range(6))],
             'weight_decay': 0.01, 'lr': learning_rate * 0.1},
            {'params': [p for n, p in model.bert.named_parameters()
                        if not any(nd in n for nd in no_decay)
                        and any(f'layer.{i}.' in n for i in range(6, 12))],
             'weight_decay': 0.01, 'lr': learning_rate},
            {'params': model.classifier.parameters(),
             'weight_decay': 0.01, 'lr': learning_rate * 5},
            {'params': [p for n, p in model.named_parameters()
                        if any(nd in n for nd in no_decay)],
             'weight_decay': 0.0, 'lr': learning_rate},
        ]
        optimizer = torch.optim.AdamW(optimizer_grouped)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    # Scheduler (Cosine Annealing with Warmup)
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    # Функция потерь
    if custom_loss_fn is not None:
        loss_fn = custom_loss_fn
        print(f"    Используем кастомную функцию потерь: {type(loss_fn).__name__}")
    else:
        loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        print(f"    Используем CrossEntropyLoss (label_smoothing={label_smoothing})")

    # 4. Цикл обучения с оценкой на VALIDATION
    print(f" Начинаем обучение...")
    best_val_f1 = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Используем кастомную или стандартную функцию потерь
            loss = loss_fn(outputs.logits, labels)

            loss.backward()

            # Gradient clipping
            if gradient_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)

            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Оценка на VALIDATION после каждой эпохи
        val_f1, val_acc = evaluate_model(model, val_loader, DEVICE)
        avg_loss = total_loss / len(train_loader)
        print(f"   Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

        # Сохраняем лучшую модель по validation F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

    # Загружаем лучшую модель
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Сохранение модели
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"    Сохранено в {save_dir}")

    print(f"\n Лучший Val Macro-F1: {best_val_f1:.4f}")

    return {
        'val_macro_f1': best_val_f1,
        'model': model,
        'tokenizer': tokenizer,
    }

## Определение 5 трансформеров для сравнения

### Почему именно эти модели?
rubert-base-cased — это золотой стандарт для русского NLP. Обучен на огромном корпусе русских текстов (Википедия, новости, форумы). Понимает русскую морфологию, контекст, иронию.

ruRoberta-large — улучшенная версия RuBERT с большим размером модели и более агрессивным маскированием при предобучении. Даёт +2-4% к качеству, но обучается в 3 раза дольше.

rubert-tiny2 — крошечная модель для быстрых экспериментов. Если нужно проверить, что пайплайн работает, — запускай её.

rubert-base-cased-sentiment-rusentiment — уже дообучена на датасете rusentiment (русские отзывы). Можно использовать как отправную точку — её веса уже "настроены" на задачу тональности.

xlm-roberta-base — мультиязычная модель, обученная на 100 языках одновременно. Интересна тем, что показывает, насколько важно использовать именно русскоязычную модель vs мультиязычную.

### Rubert-tiny2

In [ ]:
result_8 = train_transformer(
    model_name='cointegrated/rubert-tiny2',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=64,
    learning_rate=3e-5, epochs=3, model_key='rubert-tiny2',
)
transformer_results = {'rubert-tiny2': result_8}

### Rubert-base

In [ ]:
result_9 = train_transformer(
    model_name='DeepPavlov/rubert-base-cased',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='rubert-base',
)
transformer_results['rubert-base'] = result_9

### Rubert-sentiment

In [ ]:
result_10 = train_transformer(
    model_name='blanchefort/rubert-base-cased-sentiment-rusentiment',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='rubert-sentiment',
)
transformer_results['rubert-sentiment'] = result_10

### Xlm-roberta-base

In [ ]:
result_11 = train_transformer(
    model_name='FacebookAI/xlm-roberta-base',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=3, model_key='xlm-roberta-base',
)
transformer_results['xlm-roberta-base'] = result_11

### RuRoberta-large

In [ ]:
result_12 = train_transformer(
    model_name='sberbank-ai/ruRoberta-large',
    train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=8,
    learning_rate=1e-5, epochs=2, model_key='ruRoberta-large',
)
transformer_results['ruRoberta-large'] = result_12

### СВОДНАЯ ТАБЛИЦА ТРАНСФОРМЕРОВ
> Добавить блок с цитатой



In [ ]:
print("ЯЧЕЙКА 13: СВОДНАЯ ТАБЛИЦА ТРАНСФОРМЕРОВ")

print(f"\n{'Модель':<25} | {'Val Macro-F1':<12}")
print("-" * 40)
for key, res in sorted(transformer_results.items(), key=lambda x: x[1]['val_macro_f1'], reverse=True):
    print(f"{key:<25} | {res['val_macro_f1']:.4f}")

best_tf_key = max(transformer_results, key=lambda k: transformer_results[k]['val_macro_f1'])
best_tf_f1 = transformer_results[best_tf_key]['val_macro_f1']
best_tf_model = transformer_results[best_tf_key]['model']
best_tf_tokenizer = transformer_results[best_tf_key]['tokenizer']

print(f"\n Лучший: {best_tf_key} (Val F1={best_tf_f1:.4f})")

# Визуализация
plt.figure(figsize=(12, 6))
models = list(transformer_results.keys())
f1_values = [transformer_results[k]['val_macro_f1'] for k in models]
plt.bar(models, f1_values, color=['#2ecc71' if v == max(f1_values) else '#3498db' for v in f1_values])
plt.axhline(y=best_tf_f1, color='red', linestyle='--', linewidth=2, label=f'Лучший: {best_tf_f1:.4f}')
plt.xlabel('Модель'), plt.ylabel('Val Macro-F1')
plt.title('Сравнение 5 трансформеров', fontsize=14)
plt.xticks(rotation=45), plt.legend()
plt.tight_layout()
plt.show()

## Аугментация

### Аугментация нейтрального класса - Back-Translation

In [ ]:
print("BACK-TRANSLATION")

from transformers import pipeline

print(" Загрузка моделей перевода...")
translator_ru_en = pipeline('translation', model='Helsinki-NLP/opus-mt-ru-en', device=0 if torch.cuda.is_available() else -1)
translator_en_ru = pipeline('translation', model='Helsinki-NLP/opus-mt-en-ru', device=0 if torch.cuda.is_available() else -1)

def back_translate(text: str) -> str:
    try:
        en = translator_ru_en(text, max_length=128)[0]['translation_text']
        return translator_en_ru(en, max_length=128)[0]['translation_text']
    except:
        return text

# Аугментация только нейтрального класса (label=0)
neutral_mask = train_df['sentiment'] == 0
neutral_texts = train_df.loc[neutral_mask, text_col].tolist()
print(f"Исходных нейтральных: {len(neutral_texts)}")

# Создаём 2 парафраза для каждого
aug_texts, aug_labels = [], []
for i, text in enumerate(neutral_texts[:5000]):  # ограничим для скорости
    if i % 500 == 0:
        print(f"   {i}/{min(5000, len(neutral_texts))}")
    for _ in range(2):
        aug_texts.append(back_translate(text))
        aug_labels.append(0)

# Объединяем с исходными данными
augmented_train_df = pd.concat([
    train_df,
    pd.DataFrame({text_col: aug_texts, 'sentiment': aug_labels})
], ignore_index=True)

print(f"\n Аугментация завершена")
print(f"   Было: {len(train_df):,}")
print(f"   Стало: {len(augmented_train_df):,}")
print(f"   Прирост нейтральных: {len(aug_texts)}")

### Аугментация нейтрального класса - Контекстная замена

In [ ]:
print("КОНТЕКСТНАЯ ЗАМЕНА")

!pip install -q nlpaug

import nlpaug.augmenter.word as naw

context_aug = naw.ContextualWordEmbsAug(
    model_path='distilbert-base-russian-cased',
    action='substitute', aug_min=1, aug_max=3, aug_p=0.2,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print(" Аугментация нейтрального класса...")
context_aug_texts = []
for i, text in enumerate(neutral_texts[:3000]):
    if i % 500 == 0:
        print(f"   {i}/3000")
    try:
        context_aug_texts.append(context_aug.augment(text))
    except:
        context_aug_texts.append(text)

# Добавляем к аугментированному датасету
augmented_train_df = pd.concat([
    augmented_train_df,
    pd.DataFrame({text_col: context_aug_texts, 'sentiment': [0] * len(context_aug_texts)})
], ignore_index=True)

print(f"\n Финальный размер: {len(augmented_train_df):,}")

## Вычисление весов классов для аугментированного датасета

In [ ]:
print("ВЕСА КЛАССОВ")

class_counts = augmented_train_df['sentiment'].value_counts().sort_index().values
total = len(augmented_train_df)

print(f"\n Распределение:")
for cls, count in enumerate(class_counts):
    print(f"   Класс {cls} ({CLASS_NAMES[cls]}): {count:,} ({count/total*100:.1f}%)")

class_weights_np = total / (NUM_CLASSES * class_counts)
class_weights_tensor = torch.tensor(class_weights_np, dtype=torch.float32).to(DEVICE)

print(f"\n Веса: {class_weights_np.round(4)}")

## Переобучение топ-3 моделей на аугментированных данных с весами

In [ ]:
print("ПЕРЕОБУЧЕНИЕ ТОП-3 НА АУГМЕНТИРОВАННЫХ ДАННЫХ")

# Топ-3 модели по Val F1 из Ячейки 13
top_3_keys = sorted(transformer_results, key=lambda k: transformer_results[k]['val_macro_f1'], reverse=True)[:3]
print(f" Топ-3 модели: {top_3_keys}")

# =============================================================================
# Кастомная функция потерь с весами классов
# =============================================================================
class WeightedCELoss(nn.Module):
    """
    CrossEntropyLoss с весами классов для борьбы с дисбалансом.
    """
    def __init__(self, class_weights: torch.Tensor):
        super().__init__()
        self.weights = class_weights.to(DEVICE)

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(logits, labels, weight=self.weights)

# Создаём функцию потерь с весами из Ячейки 16
weighted_loss_fn = WeightedCELoss(class_weights_tensor)

print(f"\n⚖️ Используем WeightedCELoss с весами: {class_weights_np.round(4)}")

# =============================================================================
# Переобучение каждой из топ-3 моделей
# =============================================================================
augmented_results = {}

for key in top_3_keys:
    # Получаем название модели из tokenizer
    model_name = transformer_results[key]['tokenizer'].name_or_path

    print(f"\n{'=' * 60}")
    print(f"🔄 Переобучение: {key}")
    print(f"{'=' * 60}")

    # Вызываем train_transformer с кастомной функцией потерь
    result = train_transformer(
        model_name=model_name,
        train_df=augmented_train_df,  # ← АУГМЕНТИРОВАННЫЕ данные
        val_df=val_df,                # ← Validation (не test!)
        text_col=text_col,
        max_length=128,
        batch_size=16,
        learning_rate=2e-5,
        epochs=2,
        custom_loss_fn=weighted_loss_fn,  # ← ВЕСА КЛАССОВ
        model_key=f'Augmented-{key}',
    )

    augmented_results[key] = result

# =============================================================================
# Сводная таблица результатов
# =============================================================================
print(" РЕЗУЛЬТАТЫ ПЕРЕОБУЧЕНИЯ")

print(f"\n{'Модель':<30} | {'Val Macro-F1 (до)':<18} | {'Val Macro-F1 (после)':<20} | {'Прирост':<10}")
print("-" * 85)

for key in top_3_keys:
    before_f1 = transformer_results[key]['val_macro_f1']
    after_f1 = augmented_results[key]['val_macro_f1']
    improvement = after_f1 - before_f1

    print(f"{key:<30} | {before_f1:<18.4f} | {after_f1:<20.4f} | {improvement:+.4f}")

# Лучшая модель после аугментации
best_aug_key = max(augmented_results, key=lambda k: augmented_results[k]['val_macro_f1'])
best_aug_f1 = augmented_results[best_aug_key]['val_macro_f1']

print(f"\n Лучшая модель после аугментации: {best_aug_key}")
print(f"   Val Macro-F1: {best_aug_f1:.4f}")

# Сохранение лучшей модели
os.makedirs('./augmented_models', exist_ok=True)
save_path = f'./augmented_models/best_augmented_{best_aug_key}'
augmented_results[best_aug_key]['model'].save_pretrained(save_path)
augmented_results[best_aug_key]['tokenizer'].save_pretrained(save_path)
print(f"    Сохранено в {save_path}")


## Cosine Annealing with Warmup - детальное объяснение и визуализация

In [ ]:
# =============================================================================
# ЯЧЕЙКА 18: COSINE ANNEALING С WARMUP - ВИЗУАЛИЗАЦИЯ
# =============================================================================
print("\n" + "=" * 70)
print("ЯЧЕЙКА 18: COSINE ANNEALING С WARMUP")
print("=" * 70)

print("""
 Cosine Annealing with Warmup:

Этап 1: WARMUP (первые 10% шагов)
  • LR плавно растёт от 0 до целевого значения
  • Зачем: в начале обучения градиенты нестабильны

Этап 2: COSINE ANNEALING (остальные 90%)
  • LR плавно уменьшается по косинусоиде до почти 0
  • Зачем: модель аккуратно скатывается в минимум loss

Почему помогает для Нейтрального класса?
  Нейтраль — редкий класс. Без warmup модель "забывает" его
  в начале обучения. Cosine annealing в конце позволяет
  точнее настроить границу между классами.
""")

# Визуализация
LR_MAX, LR_MIN, WARMUP_RATIO = 2e-5, 1e-6, 0.1
TOTAL_STEPS = 1000
warmup_steps = int(TOTAL_STEPS * WARMUP_RATIO)

lrs = []
for step in range(TOTAL_STEPS):
    if step < warmup_steps:
        lr = LR_MAX * (step / warmup_steps)
    else:
        progress = (step - warmup_steps) / (TOTAL_STEPS - warmup_steps)
        lr = LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1 + np.cos(np.pi * progress))
    lrs.append(lr)

plt.figure(figsize=(14, 6))
plt.plot(lrs, linewidth=3, color='#3498db')
plt.axvline(x=warmup_steps, color='#e74c3c', linestyle='--', linewidth=2, label=f'Warmup end')
plt.axhline(y=LR_MAX, color='#2ecc71', linestyle=':', label=f'LR max')
plt.xlabel('Training Step', fontsize=12), plt.ylabel('Learning Rate', fontsize=12)
plt.title('Cosine Annealing with Warmup', fontsize=14, fontweight='bold')
plt.legend(), plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n В нашем коде это реализовано через:")
print(f"   get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=..., num_training_steps=...)")

## Ансамблирование топ-3 трансформеров

In [ ]:
print("АНСАМБЛЬ ТОП-3")

# Собираем топ-3 модели (используем augmented_results если есть, иначе transformer_results)
models_to_ensemble = augmented_results if augmented_results else {k: transformer_results[k] for k in top_3_keys}

# Оценка на validation
all_probs, all_labels = [], []
val_dataset_ens = SentimentDataset(
    val_df[text_col].tolist(), val_df['sentiment'].tolist(),
    best_tf_tokenizer, 128
)
val_loader_ens = DataLoader(val_dataset_ens, batch_size=16, shuffle=False)

with torch.no_grad():
    for batch in val_loader_ens:
        probs_batch = []
        for key, res in models_to_ensemble.items():
            model = res['model']
            model.eval()
            out = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE)
            )
            probs_batch.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_probs.append(np.mean(probs_batch, axis=0))
        all_labels.extend(batch['labels'].cpu().numpy())

all_probs = np.vstack(all_probs)
ensemble_preds = np.argmax(all_probs, axis=1)
ensemble_f1 = f1_score(all_labels, ensemble_preds, average='macro', zero_division=0)

print(f"\n РЕЗУЛЬТАТ АНСАМБЛЯ:")
print(f"   Val Macro-F1: {ensemble_f1:.4f}")

# Сравнение с одиночными моделями
print(f"\n Сравнение:")
for key, res in models_to_ensemble.items():
    print(f"   {key}: {res['val_macro_f1']:.4f}")
print(f"   Ансамбль: {ensemble_f1:.4f} ({ensemble_f1 - max(r['val_macro_f1'] for r in models_to_ensemble.values()):+.4f})")

## Финальный анализ ошибок и сохранение моделей

In [ ]:
print("ФИНАЛЬНЫЙ АНАЛИЗ")

# Определяем лучшую модель (ансамбль или одиночная)
best_overall_f1 = max(ensemble_f1, max(r['val_macro_f1'] for r in models_to_ensemble.values()))

if ensemble_f1 >= max(r['val_macro_f1'] for r in models_to_ensemble.values()):
    print(f" Лучшая: АНСАМБЛЬ (Val F1={ensemble_f1:.4f})")
    best_model_type = 'ensemble'
else:
    best_key = max(models_to_ensemble, key=lambda k: models_to_ensemble[k]['val_macro_f1'])
    print(f" Лучшая: {best_key} (Val F1={models_to_ensemble[best_key]['val_macro_f1']:.4f})")
    best_model_type = 'single'

# Сохранение
os.makedirs('./final_models', exist_ok=True)
if best_model_type == 'ensemble':
    for key, res in models_to_ensemble.items():
        save_path = f'./final_models/ensemble_{key}'
        res['model'].save_pretrained(save_path)
        res['tokenizer'].save_pretrained(save_path)
        print(f"   Сохранён ансамбль {key}")
else:
    save_path = f'./final_models/best_{best_key}'
    models_to_ensemble[best_key]['model'].save_pretrained(save_path)
    models_to_ensemble[best_key]['tokenizer'].save_pretrained(save_path)
    print(f"   Сохранена модель {best_key}")

# Метрика для сравнения в Ячейке 35
baseline_final_f1 = best_overall_f1
print(f"\n Базовая метрика для Ячейки 35: {baseline_final_f1:.4f}")

# Дополнительные эксперименты

## Продвинутые техники обучения

In [ ]:
BEST_MODEL_NAME = best_tf_tokenizer.name_or_path  # или transformer_results[best_tf_key]['tokenizer'].name_or_path
technique_results = {}

### Differential Learning Rates (Разные скорости обучения для слоёв)

Суть: Замороженные слои BERT (нижние) обучаются медленнее, а классификационная голова (верхние слои) — быстрее.

Нижние слои BERT уже выучили общую лингвистику, их не нужно сильно менять. Верхние слои нужно адаптировать под конкретную задачу.

In [ ]:
result_21 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    use_differential_lr=True, model_key='21. Differential LR',
)
technique_results['21. Differential LR'] = result_21['val_macro_f1']

### Label Smoothing (Сглаживание меток)

Суть: Вместо жёстких меток [0, 1, 0] используем мягкие [0.1, 0.8, 0.1]. Это не даёт модели быть слишком уверенной и улучшает обобщение.

Модель становится менее "уверенной" в своих предсказаниях, что особенно полезно для размытого класса "Нейтраль".

In [ ]:
result_22 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    label_smoothing=0.1, model_key='22. Label Smoothing',
)
technique_results['22. Label Smoothing'] = result_22['val_macro_f1']

### Gradient Clipping (Обрезка градиентов)

Суть: Ограничивает максимальное значение градиентов, предотвращая "взрыв" градиентов.

Стабилизирует обучение, особенно на длинных текстах.

In [ ]:
result_23 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    gradient_clip_norm=0.5, model_key='23. Gradient Clipping',
)
technique_results['23. Gradient Clipping'] = result_23['val_macro_f1']

## Архитектурные модификации

###  Custom Classifier Head (Добавление дополнительных слоёв поверх BERT)


Суть: Вместо одного классификационного слоя добавляем несколько слоёв с dropout и активациями.

Даёт модели большеcapacity для сложных паттернов.

In [ ]:
result_24 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    custom_model_class=CustomHeadModel, custom_model_kwargs={},
    model_key='24. Custom MLP Head',
)
technique_results['24. Custom MLP Head'] = result_24['val_macro_f1']

### Использование [CLS] + Mean Pooling

Суть: Вместо использования только [CLS] токена, усредняем все токены и конкатенируем с [CLS].

[CLS] может не захватывать всю информацию, особенно для длинных текстов.

In [ ]:
result_25 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    custom_model_class=MeanPoolModel, custom_model_kwargs={},
    model_key='25. Mean Pooling + CLS',
)
technique_results['25. Mean Pooling + CLS'] = result_25['val_macro_f1']

## Работа с данными

### Active Learning (Активное обучение)

Суть: Модель сама выбирает, какие примеры ей нужны для обучения.

Как работает:
Обучаем модель на начальном датасете
Модель предсказывает на неразмеченных данных
Выбираешь примеры, где модель наименее уверена (entropy высокая)
Размечаешь их вручную или через аугментацию
Дообучаешь модел

In [ ]:
print("ACTIVE LEARNING")

# Шаг 1: Обучение на 50% данных
half = len(train_df) // 2
train_half = train_df.iloc[:half]

result_26_init = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_half, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=1, model_key='26. Active Learning (init)',
)

# Шаг 2: Поиск сложных примеров
model_26_init = result_26_init['model']
model_26_init.eval()

hard_texts, hard_labels = [], []
remaining = train_df.iloc[half:]

with torch.no_grad():
    for i in range(0, len(remaining), 16):
        batch = remaining.iloc[i:i+16]
        texts = batch[text_col].tolist()
        labels = batch['sentiment'].tolist()
        enc = result_26_init['tokenizer'](
            texts, padding=True, truncation=True, max_length=128, return_tensors='pt'
        ).to(DEVICE)
        probs = F.softmax(model_26_init(**enc).logits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
        top_k = max(1, int(len(texts) * 0.3))
        indices = torch.topk(entropy, min(top_k, len(entropy))).indices
        hard_texts.extend([texts[idx] for idx in indices.cpu()])
        hard_labels.extend([labels[idx] for idx in indices.cpu()])

hard_df = pd.DataFrame({text_col: hard_texts, 'sentiment': hard_labels})
combined_df = pd.concat([train_half, hard_df], ignore_index=True)

# Шаг 3: Дообучение
result_26 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=combined_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=1e-5, epochs=1, model_key='26. Active Learning',
)
technique_results['26. Active Learning'] = result_26['val_macro_f1']

### Сurriculum Learning (Обучение по учебному плану)

Суть: Начинаем обучение с простых примеров, постепенно переходя к сложным.


Сортируем датасет по "сложности" (длина текста, количество стоп-слов и т.д.)
Первые эпохи — только простые примеры
Последние эпохи — все примеры

In [ ]:
print("CURRICULUM LEARNING")

# Сортируем по длине текста
lengths = [len(t) for t in train_df[text_col].tolist()]
sorted_idx = np.argsort(lengths)
sorted_texts = [train_df[text_col].tolist()[i] for i in sorted_idx]
sorted_labels = [train_df['sentiment'].tolist()[i] for i in sorted_idx]

tokenizer_cur = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
model_cur = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_NAME, num_labels=3).to(DEVICE)
optimizer_cur = torch.optim.AdamW(model_cur.parameters(), lr=2e-5)

for epoch in range(2):
    progress = (epoch + 1) / 2
    cur_size = int(len(sorted_texts) * (0.3 + 0.7 * progress))

    dataset_cur = SentimentDataset(sorted_texts[:cur_size], sorted_labels[:cur_size], tokenizer_cur, 128)
    loader_cur = DataLoader(dataset_cur, batch_size=16, shuffle=True)

    total_steps = len(loader_cur)
    scheduler_cur = get_cosine_schedule_with_warmup(optimizer_cur, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    model_cur.train()
    for batch in loader_cur:
        optimizer_cur.zero_grad()
        out = model_cur(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE), labels=batch['labels'].to(DEVICE))
        out.loss.backward()
        optimizer_cur.step()
        scheduler_cur.step()

    val_f1, _ = evaluate_model(model_cur, DataLoader(SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_cur, 128), batch_size=16), DEVICE)
    print(f"   Epoch {epoch+1}/2 | Samples: {cur_size} | Val F1: {val_f1:.4f}")

final_f1_27, _ = evaluate_model(model_cur, DataLoader(SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_cur, 128), batch_size=16), DEVICE)
technique_results['27. Curriculum Learning'] = final_f1_27
print(f" Val Macro-F1: {final_f1_27:.4f}")

### Mixup для текстов

Суть: Создаём синтетические примеры путём интерполяции эмбеддингов двух разных текстов.

Модель видит больше разнообразных примеров и учится обобщать.

In [ ]:
result_28 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    mixup_alpha=0.2, model_key='28. Mixup',
)
technique_results['28. Mixup'] = result_28['val_macro_f1']

## Постобработка и инференс

### Threshold Tuning (Настройка порогов)

Суть: Вместо использования argmax для предсказания, настраиваем пороги для каждого класса отдельно.

Если модель систематически недооценивает один класс, можно снизить порог для него.

In [ ]:
print("THRESHOLD TUNING")

result_29_init = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2, model_key='29. Threshold Tuning (train)',
)

# Поиск оптимальных порогов на validation
model_29 = result_29_init['model']
tokenizer_29 = result_29_init['tokenizer']

val_dataset_29 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), tokenizer_29, 128)
val_loader_29 = DataLoader(val_dataset_29, batch_size=16, shuffle=False)

model_29.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for batch in val_loader_29:
        out = model_29(input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
        all_probs.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

all_probs = np.vstack(all_probs)
all_labels = np.array(all_labels)

best_thresholds = [0.5, 0.5, 0.5]
for c in range(3):
    best_f1_c, best_t = 0, 0.5
    for t in np.arange(0.3, 0.7, 0.01):
        preds = (all_probs[:, c] >= t).astype(int)
        true_c = (all_labels == c).astype(int)
        tp = np.sum((preds == 1) & (true_c == 1))
        fp = np.sum((preds == 1) & (true_c == 0))
        fn = np.sum((preds == 0) & (true_c == 1))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_c = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        if f1_c > best_f1_c:
            best_f1_c, best_t = f1_c, t
    best_thresholds[c] = best_t

# Оценка с порогами
final_preds = []
for prob in all_probs:
    adj = prob.copy()
    for c, t in enumerate(best_thresholds):
        if prob[c] < t:
            adj[c] = 0
    final_preds.append(np.argmax(adj) if adj.sum() > 0 else np.argmax(prob))

f1_29 = f1_score(all_labels, final_preds, average='macro', zero_division=0)
technique_results['29. Threshold Tuning'] = f1_29
print(f" Val Macro-F1: {f1_29:.4f} | Пороги: {[round(t, 2) for t in best_thresholds]}")

### Ensemble with Different Seeds (Ансамбль с разными seed)

Суть: Обучаем одну и ту же модель 3-5 раз с разными random seed и усредняем предсказания.

Разные seed дают разные локальные минимумы, ансамбль снижает variance.

In [ ]:
print("ENSEMBLE WITH SEEDS")

seeds = [42, 1337, 2024]
models_30 = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    result = train_transformer(
        model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
        text_col=text_col, max_length=128, batch_size=16,
        learning_rate=2e-5, epochs=1, model_key=f'30. Seed={seed}',
    )
    models_30.append(result)

# Ансамбль
all_probs_list, all_labels_30 = [], []
val_dataset_30 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), models_30[0]['tokenizer'], 128)
val_loader_30 = DataLoader(val_dataset_30, batch_size=16, shuffle=False)

with torch.no_grad():
    for batch in val_loader_30:
        probs_batch = []
        for res in models_30:
            res['model'].eval()
            out = res['model'](input_ids=batch['input_ids'].to(DEVICE), attention_mask=batch['attention_mask'].to(DEVICE))
            probs_batch.append(F.softmax(out.logits, dim=-1).cpu().numpy())
        all_probs_list.append(np.mean(probs_batch, axis=0))
        all_labels_30.extend(batch['labels'].cpu().numpy())

final_preds_30 = np.argmax(np.vstack(all_probs_list), axis=1)
f1_30 = f1_score(all_labels_30, final_preds_30, average='macro', zero_division=0)
technique_results['30. Ensemble with Seeds'] = f1_30
print(f" Val Macro-F1: {f1_30:.4f}")

## Дополнительные подходы

### Knowledge Distillation (Дистилляция знаний)

Суть: Обучаем маленькую модель (student) предсказывать то же, что и большая (teacher).

Student наследует знания teacher, но работает быстрее.

In [ ]:
print("KNOWLEDGE DISTILLATION")

print(" Пропускаем (требует много памяти)")
technique_results['31. Knowledge Distillation'] = baseline_final_f1
print(f" Val Macro-F1: {baseline_final_f1:.4f} (пропущено)")

### Multi-Task Learning (Многозадачное обучение)

Суть: Обучаем модель одновременно на нескольких задачах (например, тональность + длина текста + категория товара).

Модель учится более общим представлениям текста.

In [ ]:
print("MULTI-TASK LEARNING")

def get_aux_label(text):
    l = len(str(text))
    if l < 100: return 0
    elif l < 200: return 1
    elif l < 300: return 2
    elif l < 400: return 3
    return 4

mt_train = MTDataset(
    train_df[text_col].tolist(),
    train_df['sentiment'].tolist(),
    [get_aux_label(t) for t in train_df[text_col].tolist()],
    AutoTokenizer.from_pretrained(BEST_MODEL_NAME),
)
mt_loader = DataLoader(mt_train, batch_size=16, shuffle=True)

model_32 = MultiTaskModel(BEST_MODEL_NAME).to(DEVICE)
optimizer_32 = torch.optim.AdamW(model_32.parameters(), lr=2e-5)
total_steps = len(mt_loader) * 2
scheduler_32 = get_cosine_schedule_with_warmup(optimizer_32, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

for epoch in range(2):
    model_32.train()
    for batch in mt_loader:
        optimizer_32.zero_grad()
        out = model_32(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            sent_labels=batch['sent_labels'].to(DEVICE),
            aux_labels=batch['aux_labels'].to(DEVICE),
        )
        if out.loss is not None:
            out.loss.backward()
        optimizer_32.step()
        scheduler_32.step()

val_dataset_32 = SentimentDataset(val_df[text_col].tolist(), val_df['sentiment'].tolist(), AutoTokenizer.from_pretrained(BEST_MODEL_NAME), 128)
val_loader_32 = DataLoader(val_dataset_32, batch_size=16, shuffle=False)
f1_32, _ = evaluate_model(model_32, val_loader_32, DEVICE)
technique_results['32. Multi-Task Learning'] = f1_32
print(f" Val Macro-F1: {f1_32:.4f}")

### Adversarial Training (Состязательное обучение)

Суть: Добавляем небольшой шум к эмбеддингам, чтобы модель была устойчивее.

Модель учится не полагаться на хрупкие паттерны

In [ ]:
result_33 = train_transformer(
    model_name=BEST_MODEL_NAME, train_df=train_df, val_df=val_df,
    text_col=text_col, max_length=128, batch_size=16,
    learning_rate=2e-5, epochs=2,
    use_fgm=True, fgm_epsilon=1.0, model_key='33. Adversarial Training',
)
technique_results['33. Adversarial Training'] = result_33['val_macro_f1']

## Сравнение моделей из ячеек 15-27

In [ ]:
print("СРАВНЕНИЕ ТЕХНИК")

sorted_tech = sorted(technique_results.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'Техника':<35} | {'Val Macro-F1':<12} | {'Прирост':<10}")
print("-" * 65)
for name, f1 in sorted_tech:
    print(f"{name:<35} | {f1:<12.4f} | {f1 - baseline_final_f1:+.4f}")

best_tech_name, best_tech_f1 = sorted_tech[0]
print(f"\n ЛУЧШАЯ ТЕХНИКА: {best_tech_name} (Val F1: {best_tech_f1:.4f})")

# Визуализация
plt.figure(figsize=(14, 8))
names = [x[0] for x in sorted_tech]
values = [x[1] for x in sorted_tech]
colors = ['#2ecc71' if v > baseline_final_f1 else '#e74c3c' for v in values]

plt.barh(names, values, color=colors)
plt.axvline(x=baseline_final_f1, color='blue', linestyle='--', linewidth=2, label=f'Baseline: {baseline_final_f1:.4f}')
plt.xlabel('Val Macro-F1', fontsize=12)
plt.title('Сравнение техник докрутки (Ячейки 21-33)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.legend()
for i, v in enumerate(values):
    plt.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### Ансамль vs ЛУЧШАЯ ТЕХНИКА

In [ ]:
print("ФИНАЛЬНОЕ СРАВНЕНИЕ")

print(f"\n Базовая модель + аугментация + ансамбль: {baseline_final_f1:.4f}")
print(f" Лучшая техника докрутки: {best_tech_f1:.4f}")
print(f" Разница: {best_tech_f1 - baseline_final_f1:+.4f}")

diff = best_tech_f1 - baseline_final_f1
print("-" * 80)
if diff > 0.02:
    print(f" ОТЛИЧНЫЙ РЕЗУЛЬТАТ! Прирост +{diff:.4f}")
elif diff > 0.01:
    print(f" ХОРОШИЙ РЕЗУЛЬТАТ! Прирост +{diff:.4f}")
elif diff > 0:
    print(f" Небольшой прирост: +{diff:.4f}")
else:
    print(f"ℹ Базовая модель оказалась лучше ({diff:+.4f})")

# Финальный график
plt.figure(figsize=(10, 6))
categories = ['Ячейка 20\n(Базовая + Аугментация)', f'Ячейка 34\n({best_tech_name})']
vals = [baseline_final_f1, best_tech_f1]
colors_bar = ["#3498db", "#2ecc71"]

bars = plt.bar(categories, vals, color=colors_bar, width=0.6)
plt.ylabel("Val Macro-F1", fontsize=12)
plt.title("Финальное сравнение: Ячейка 20 vs Ячейка 34", fontsize=14, fontweight="bold")
plt.ylim(min(vals) - 0.05, max(vals) + 0.05)

for bar, val in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

print(f"   Лучший Val Macro-F1: {max(baseline_final_f1, best_tech_f1):.4f}")

## Эксперимент 17 — исторический RuBERT
[Исходный Colab](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/ru_sentiment_research_FINAL.ipynb)

Архивный сценарий восстановления и обучения; старые чекпоинты отсутствуют. Не запускать блоки восстановления без весов.

# Эксперимент № 17. RuBERT (Fine-tuning)

In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 1. ПОДГОТОВКА ОКРУЖЕНИЯ
# ============================================================

import sys
import subprocess

# 1. Устанавливаем библиотеки
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers",
    "accelerate",
    "safetensors"
])

# 2. Импортируем библиотеки
import os
import torch
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# 3. Проверяем исходные данные
required = [
    "X_train_new",
    "y_train_new",
    "X_val_new",
    "y_val_new",
    "CLASS_NAMES_NEW"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )

assert len(X_train_new) == 203357
assert len(X_val_new) == 20890
assert len(CLASS_NAMES_NEW) == 3

# 4. Проверяем GPU
DEVICE_RUBERT = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("ЭКСПЕРИМЕНТ 17 — RuBERT")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Устройство:", DEVICE_RUBERT)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )
    print(
        "Видеопамять (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        )
    )
else:
    raise RuntimeError(
        "Для обучения RuBERT нужен GPU. "
        "В Colab выбери Runtime → Change runtime type → T4 GPU."
    )

# 5. Настройки модели
MODEL_NAME_RUBERT = "DeepPavlov/rubert-base-cased"

NUM_LABELS_RUBERT = 3

ID2LABEL_RUBERT = {
    0: "Neutral",
    1: "Positive",
    2: "Negative"
}

LABEL2ID_RUBERT = {
    name: idx
    for idx, name in ID2LABEL_RUBERT.items()
}

# 6. Загружаем токенизатор
print("\nЗагружаем токенизатор...")

tokenizer_rubert = AutoTokenizer.from_pretrained(
    MODEL_NAME_RUBERT
)

# 7. Загружаем модель
print("Загружаем RuBERT...")

model_rubert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_RUBERT,
    num_labels=NUM_LABELS_RUBERT,
    id2label=ID2LABEL_RUBERT,
    label2id=LABEL2ID_RUBERT
)

# Пока оставляем модель на CPU.
# На GPU перенесём перед обучением.

# 8. Проверяем токенизацию
example_text = "Мне очень понравился этот товар!"

encoded_example = tokenizer_rubert(
    example_text,
    truncation=True,
    max_length=256
)

print("\n" + "=" * 60)
print("ПРОВЕРКА")
print("=" * 60)

print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("Модель:", MODEL_NAME_RUBERT)
print("Классы:", ID2LABEL_RUBERT)
print("Размер словаря токенизатора:", len(tokenizer_rubert))
print("Количество параметров модели:",
      f"{sum(p.numel() for p in model_rubert.parameters()):,}")

print("\nТестовый текст:", example_text)
print(
    "Токены:",
    tokenizer_rubert.convert_ids_to_tokens(
        encoded_example["input_ids"]
    )
)

print("\n✅ RuBERT загружен и готов к подготовке данных!")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 2. ТОКЕНИЗАЦИЯ И DATALOADER
# ============================================================

import os
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding

# 1. Настройки
MAX_LEN_RUBERT = 128
BATCH_SIZE_RUBERT = 8

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

# 2. Проверяем данные
assert len(X_train_new) == 203357
assert len(X_val_new) == 20890

assert len(X_train_new) == len(y_train_new)
assert len(X_val_new) == len(y_val_new)

assert set(np.unique(y_train_new)) == {0, 1, 2}
assert set(np.unique(y_val_new)) == {0, 1, 2}

# 3. Создаём Dataset
class RuBERTDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length):

        self.labels = np.asarray(
            labels, dtype=np.int64
        )

        print(
            f"Токенизация {len(texts):,} текстов..."
        )

        # Токенизируем заранее, чтобы не делать
        # это повторно на каждой эпохе
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_length,
            padding=False,
            return_attention_mask=True
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": int(self.labels[idx])
        }


# 4. Токенизируем Train
print("=" * 60)
print("ПОДГОТОВКА TRAIN")
print("=" * 60)

train_dataset_rubert = RuBERTDataset(
    X_train_new,
    y_train_new,
    tokenizer_rubert,
    MAX_LEN_RUBERT
)

# 5. Токенизируем Validation
print("\n" + "=" * 60)
print("ПОДГОТОВКА VALIDATION")
print("=" * 60)

val_dataset_rubert = RuBERTDataset(
    X_val_new,
    y_val_new,
    tokenizer_rubert,
    MAX_LEN_RUBERT
)

# 6. Динамический padding
collator_rubert = DataCollatorWithPadding(
    tokenizer=tokenizer_rubert,
    padding=True,
    return_tensors="pt"
)

# 7. DataLoader
train_loader_rubert = DataLoader(
    train_dataset_rubert,
    batch_size=BATCH_SIZE_RUBERT,
    shuffle=True,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

val_loader_rubert = DataLoader(
    val_dataset_rubert,
    batch_size=BATCH_SIZE_RUBERT,
    shuffle=False,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

# 8. Проверяем батч
batch_rubert = next(iter(train_loader_rubert))

print("\n" + "=" * 60)
print("ПРОВЕРКА DATALOADER")
print("=" * 60)

print("Train samples:", len(train_dataset_rubert))
print("Validation samples:", len(val_dataset_rubert))

print("Train batches:", len(train_loader_rubert))
print("Validation batches:", len(val_loader_rubert))

print(
    "input_ids:",
    tuple(batch_rubert["input_ids"].shape)
)

print(
    "attention_mask:",
    tuple(batch_rubert["attention_mask"].shape)
)

print(
    "labels:",
    tuple(batch_rubert["labels"].shape)
)

print(
    "Метки:",
    torch.unique(batch_rubert["labels"]).tolist()
)

assert batch_rubert["input_ids"].shape[0] == BATCH_SIZE_RUBERT

assert batch_rubert["input_ids"].shape[1] <= MAX_LEN_RUBERT

assert batch_rubert["input_ids"].shape == batch_rubert["attention_mask"].shape

print("\n✅ Данные для RuBERT подготовлены!")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 3. ТЕСТ СКОРОСТИ И ВИДЕОПАМЯТИ
# ============================================================

import time
import gc
import torch

from itertools import islice
from torch.optim import AdamW

# 1. Настройки теста
TEST_STEPS_RUBERT = 10
TEST_LR_RUBERT = 2e-5

assert DEVICE_RUBERT.type == "cuda", (
    "Для теста требуется GPU"
)

# 2. Переносим модель на GPU
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_rubert = model_rubert.to(DEVICE_RUBERT)
model_rubert.train()

optimizer_test_rubert = AdamW(
    model_rubert.parameters(),
    lr=TEST_LR_RUBERT
)

print("=" * 60)
print("ТЕСТ ПРОИЗВОДИТЕЛЬНОСТИ RuBERT")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Batch size:", BATCH_SIZE_RUBERT)
print("Max length:", MAX_LEN_RUBERT)
print("Тестовых шагов:", TEST_STEPS_RUBERT)

# 3. Пробные шаги обучения
step_times = []
step_losses = []

for step, batch in enumerate(
    islice(train_loader_rubert, TEST_STEPS_RUBERT),
    start=1
):
    batch = {
        key: value.to(DEVICE_RUBERT)
        for key, value in batch.items()
    }

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    optimizer_test_rubert.zero_grad(set_to_none=True)

    outputs = model_rubert(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer_test_rubert.step()

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    step_times.append(elapsed)
    step_losses.append(loss.item())

    print(
        f"Шаг {step:02d}/{TEST_STEPS_RUBERT} | "
        f"Loss: {loss.item():.4f} | "
        f"Время: {elapsed:.2f} сек"
    )

# 4. Результаты
# Первый шаг исключаем из оценки средней скорости,
# поскольку он может включать дополнительную инициализацию.
measured_times = (
    step_times[1:]
    if len(step_times) > 1
    else step_times
)

avg_step_time = sum(measured_times) / len(measured_times)

estimated_epoch_hours = (
    avg_step_time * len(train_loader_rubert) / 3600
)

peak_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ ТЕСТА")
print("=" * 60)

print(f"Среднее время шага: {avg_step_time:.2f} сек")
print(f"Пиковая память GPU: {peak_memory_gb:.2f} GB")

print(
    f"Оценка времени одной эпохи: "
    f"{estimated_epoch_hours:.2f} ч"
)

print(
    "\nЭто приблизительная оценка только обучения. "
    "Валидация и сохранение модели потребуют "
    "дополнительного времени."
)

# 5. Удаляем тестовый оптимизатор
del optimizer_test_rubert
gc.collect()
torch.cuda.empty_cache()

print("\n✅ Тест производительности завершён!")
print(
    "Внимание: модель после теста изменила веса. "
    "Перед основным обучением загрузим исходную модель заново."
)


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 4. КОНФИГУРАЦИЯ ОБУЧЕНИЯ
# ============================================================

import os
import gc
import math
import random
import numpy as np
import torch

from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

# 1. Настройки эксперимента
SEED_RUBERT = 42

EPOCHS_RUBERT = 3
LR_RUBERT = 2e-5

BATCH_SIZE_RUBERT = 8
GRAD_ACCUM_RUBERT = 4

WEIGHT_DECAY_RUBERT = 0.01
WARMUP_RATIO_RUBERT = 0.1
MAX_GRAD_NORM_RUBERT = 1.0

SAVE_EVERY_RUBERT = 500

# 2. Папка эксперимента
EXP17_DIR = os.path.join(
    PROJECT_DIR,
    "experiment_17_rubert"
)

os.makedirs(EXP17_DIR, exist_ok=True)

BEST_MODEL_PATH_RUBERT = os.path.join(
    EXP17_DIR,
    "best_model.pt"
)

CHECKPOINT_PATH_RUBERT = os.path.join(
    EXP17_DIR,
    "latest_checkpoint.pt"
)

HISTORY_PATH_RUBERT = os.path.join(
    EXP17_DIR,
    "training_history.csv"
)

# 3. Фиксируем random seed
random.seed(SEED_RUBERT)
np.random.seed(SEED_RUBERT)

torch.manual_seed(SEED_RUBERT)
torch.cuda.manual_seed_all(SEED_RUBERT)

# 4. Удаляем модель после теста
if "model_rubert" in globals():
    del model_rubert

gc.collect()
torch.cuda.empty_cache()

# 5. Восстанавливаем исходные веса RuBERT
print("Загружаем исходную модель RuBERT...")

model_rubert = (
    AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME_RUBERT,
        num_labels=NUM_LABELS_RUBERT,
        id2label=ID2LABEL_RUBERT,
        label2id=LABEL2ID_RUBERT
    )
)

model_rubert = model_rubert.to(DEVICE_RUBERT)

# 6. Создаём optimizer
optimizer_rubert = AdamW(
    model_rubert.parameters(),
    lr=LR_RUBERT,
    weight_decay=WEIGHT_DECAY_RUBERT
)

# 7. Рассчитываем количество шагов
STEPS_PER_EPOCH_RUBERT = math.ceil(
    len(train_loader_rubert) / GRAD_ACCUM_RUBERT
)

TOTAL_STEPS_RUBERT = (
    STEPS_PER_EPOCH_RUBERT * EPOCHS_RUBERT
)

WARMUP_STEPS_RUBERT = int(
    TOTAL_STEPS_RUBERT * WARMUP_RATIO_RUBERT
)

# 8. Scheduler
scheduler_rubert = get_linear_schedule_with_warmup(
    optimizer_rubert,
    num_warmup_steps=WARMUP_STEPS_RUBERT,
    num_training_steps=TOTAL_STEPS_RUBERT
)

# 9. Счётчики обучения
start_epoch_rubert = 0
global_step_rubert = 0
best_f1_rubert = -1.0

# 10. Выводим конфигурацию
print("\n" + "=" * 60)
print("ЭКСПЕРИМЕНТ № 17 — КОНФИГУРАЦИЯ")
print("=" * 60)

print("Модель:", MODEL_NAME_RUBERT)
print("Устройство:", DEVICE_RUBERT)

print("\nЭпох:", EPOCHS_RUBERT)
print("Batch size:", BATCH_SIZE_RUBERT)
print("Gradient accumulation:", GRAD_ACCUM_RUBERT)

print(
    "Effective batch size:",
    BATCH_SIZE_RUBERT * GRAD_ACCUM_RUBERT
)

print("Learning rate:", LR_RUBERT)
print("Weight decay:", WEIGHT_DECAY_RUBERT)

print("\nШагов оптимизатора на эпоху:",
      STEPS_PER_EPOCH_RUBERT)

print("Всего шагов оптимизатора:",
      TOTAL_STEPS_RUBERT)

print("Warmup steps:", WARMUP_STEPS_RUBERT)

print("\nСохранение каждые",
      SAVE_EVERY_RUBERT,
      "шагов")

print("Папка:", EXP17_DIR)

print("\nGPU memory allocated:",
      round(
          torch.cuda.memory_allocated() / 1024**3,
          2
      ),
      "GB"
)

print("\n✅ Конфигурация готова!")
print("Обучение ещё не запущено.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 5. ОБУЧЕНИЕ С CHECKPOINT И ВОССТАНОВЛЕНИЕМ
# ============================================================

import os
import gc
import csv
import math
import random
import time
import numpy as np
import torch

from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score

# ------------------------------------------------------------
# 1. Вспомогательные функции
# ------------------------------------------------------------

def make_train_loader_rubert(epoch):
    """
    Фиксированный порядок перемешивания для каждой эпохи.
    Позволяет пропускать уже обработанные батчи
    после восстановления checkpoint.
    """
    generator = torch.Generator()
    generator.manual_seed(SEED_RUBERT + epoch)

    return DataLoader(
        train_dataset_rubert,
        batch_size=BATCH_SIZE_RUBERT,
        shuffle=True,
        generator=generator,
        collate_fn=collator_rubert,
        num_workers=0,
        pin_memory=True
    )


def save_checkpoint_rubert(
    epoch,
    next_batch_idx,
    global_step,
    best_f1,
    epoch_loss_sum,
    epoch_loss_count
):
    """
    Сохраняем полное состояние обучения.
    Запись через временный файл снижает риск
    повреждения основного checkpoint.
    """

    checkpoint = {
        "epoch": epoch,
        "next_batch_idx": next_batch_idx,
        "global_step": global_step,
        "best_f1": best_f1,
        "epoch_loss_sum": epoch_loss_sum,
        "epoch_loss_count": epoch_loss_count,
        "model_state_dict": model_rubert.state_dict(),
        "optimizer_state_dict": optimizer_rubert.state_dict(),
        "scheduler_state_dict": scheduler_rubert.state_dict(),
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": torch.cuda.get_rng_state_all()
    }

    temp_path = CHECKPOINT_PATH_RUBERT + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, CHECKPOINT_PATH_RUBERT)


def evaluate_rubert():

    model_rubert.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0

    with torch.inference_mode():

        for batch in val_loader_rubert:

            batch = {
                key: value.to(DEVICE_RUBERT)
                for key, value in batch.items()
            }

            outputs = model_rubert(**batch)

            total_loss += outputs.loss.item()

            predictions = outputs.logits.argmax(dim=1)

            all_true.extend(
                batch["labels"].cpu().numpy()
            )

            all_pred.extend(
                predictions.cpu().numpy()
            )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro"
    )

    accuracy = accuracy_score(
        all_true,
        all_pred
    )

    avg_loss = total_loss / len(val_loader_rubert)

    return avg_loss, macro_f1, accuracy


def save_best_model_rubert():

    temp_path = BEST_MODEL_PATH_RUBERT + ".tmp"

    torch.save(
        model_rubert.state_dict(),
        temp_path
    )

    os.replace(temp_path, BEST_MODEL_PATH_RUBERT)


def append_history_rubert(row):

    file_exists = os.path.exists(HISTORY_PATH_RUBERT)

    with open(
        HISTORY_PATH_RUBERT,
        "a",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=[
                "epoch",
                "train_loss",
                "val_loss",
                "macro_f1",
                "accuracy",
                "global_step"
            ]
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)


# ------------------------------------------------------------
# 2. Проверяем наличие checkpoint
# ------------------------------------------------------------

start_epoch = 0
start_batch_idx = 0
global_step = 0
best_f1 = -1.0

resume_loss_sum = 0.0
resume_loss_count = 0

if os.path.exists(CHECKPOINT_PATH_RUBERT):

    print("Найден checkpoint. Восстанавливаем обучение...")

    checkpoint = torch.load(
        CHECKPOINT_PATH_RUBERT,
        map_location="cpu",
        weights_only=False
    )

    model_rubert.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer_rubert.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler_rubert.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"]
    start_batch_idx = checkpoint["next_batch_idx"]
    global_step = checkpoint["global_step"]
    best_f1 = checkpoint["best_f1"]

    resume_loss_sum = checkpoint.get(
        "epoch_loss_sum", 0.0
    )

    resume_loss_count = checkpoint.get(
        "epoch_loss_count", 0
    )

    torch.set_rng_state(
        checkpoint["torch_rng_state"]
    )

    np.random.set_state(
        checkpoint["numpy_rng_state"]
    )

    random.setstate(
        checkpoint["python_rng_state"]
    )

    if torch.cuda.is_available():
        torch.cuda.set_rng_state_all(
            checkpoint["cuda_rng_state"]
        )

    del checkpoint
    gc.collect()

    print(
        f"Продолжаем с эпохи {start_epoch + 1}, "
        f"батча {start_batch_idx + 1}, "
        f"optimizer step {global_step}"
    )

else:
    print("Checkpoint не найден. Начинаем обучение с нуля.")


# ------------------------------------------------------------
# 3. Основной цикл обучения
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("СТАРТ ОБУЧЕНИЯ RuBERT")
print("=" * 60)

print("Эпох:", EPOCHS_RUBERT)
print("Всего optimizer steps:", TOTAL_STEPS_RUBERT)
print("Сохранение каждые:", SAVE_EVERY_RUBERT)

model_rubert.to(DEVICE_RUBERT)

training_start_time = time.time()

for epoch in range(start_epoch, EPOCHS_RUBERT):

    model_rubert.train()

    train_loader_epoch = make_train_loader_rubert(epoch)

    if epoch == start_epoch:
        skip_batches = start_batch_idx
        epoch_loss_sum = resume_loss_sum
        epoch_loss_count = resume_loss_count
    else:
        skip_batches = 0
        epoch_loss_sum = 0.0
        epoch_loss_count = 0

    optimizer_rubert.zero_grad(set_to_none=True)

    epoch_start_time = time.time()

    print(
        f"\n{'=' * 60}\n"
        f"ЭПОХА {epoch + 1}/{EPOCHS_RUBERT}\n"
        f"{'=' * 60}"
    )

    for batch_idx, batch in enumerate(train_loader_epoch):

        if batch_idx < skip_batches:
            continue

        batch = {
            key: value.to(DEVICE_RUBERT)
            for key, value in batch.items()
        }

        outputs = model_rubert(**batch)

        raw_loss = outputs.loss

        # Корректируем накопление градиентов
        # для последней неполной группы батчей.
        group_start = (
            batch_idx // GRAD_ACCUM_RUBERT
        ) * GRAD_ACCUM_RUBERT

        group_size = min(
            GRAD_ACCUM_RUBERT,
            len(train_loader_epoch) - group_start
        )

        loss = raw_loss / group_size

        loss.backward()

        epoch_loss_sum += raw_loss.item()
        epoch_loss_count += 1

        is_accumulation_step = (
            (batch_idx + 1) % GRAD_ACCUM_RUBERT == 0
        )

        is_last_batch = (
            batch_idx + 1 == len(train_loader_epoch)
        )

        if is_accumulation_step or is_last_batch:

            torch.nn.utils.clip_grad_norm_(
                model_rubert.parameters(),
                MAX_GRAD_NORM_RUBERT
            )

            optimizer_rubert.step()
            scheduler_rubert.step()

            optimizer_rubert.zero_grad(set_to_none=True)

            global_step += 1

            # Логирование
            if global_step % 100 == 0:

                avg_loss = (
                    epoch_loss_sum
                    / max(epoch_loss_count, 1)
                )

                elapsed_min = (
                    time.time() - epoch_start_time
                ) / 60

                print(
                    f"Epoch {epoch + 1} | "
                    f"Step {global_step}/{TOTAL_STEPS_RUBERT} | "
                    f"Loss {avg_loss:.4f} | "
                    f"{elapsed_min:.1f} мин"
                )

            # Автоматическое сохранение
            if global_step % SAVE_EVERY_RUBERT == 0:

                save_checkpoint_rubert(
                    epoch=epoch,
                    next_batch_idx=batch_idx + 1,
                    global_step=global_step,
                    best_f1=best_f1,
                    epoch_loss_sum=epoch_loss_sum,
                    epoch_loss_count=epoch_loss_count
                )

                print(
                    f"💾 Checkpoint сохранён: "
                    f"step {global_step}"
                )

    # --------------------------------------------------------
    # 4. Валидация после эпохи
    # --------------------------------------------------------

    print(f"\nВалидация эпохи {epoch + 1}...")

    val_loss, macro_f1, accuracy = evaluate_rubert()

    train_loss = (
        epoch_loss_sum / max(epoch_loss_count, 1)
    )

    print("\nРезультаты:")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Macro-F1: {macro_f1:.6f}")
    print(f"Accuracy: {accuracy:.6f}")

    # Сохраняем лучшую модель
    if macro_f1 > best_f1:

        best_f1 = macro_f1

        save_best_model_rubert()

        print(
            f"🏆 Новая лучшая модель! "
            f"Macro-F1 = {best_f1:.6f}"
        )

    # История эксперимента
    append_history_rubert({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "global_step": global_step
    })

    # Сохраняем состояние на границе эпох
    save_checkpoint_rubert(
        epoch=epoch + 1,
        next_batch_idx=0,
        global_step=global_step,
        best_f1=best_f1,
        epoch_loss_sum=0.0,
        epoch_loss_count=0
    )

    print(
        f"\nЭпоха {epoch + 1} завершена. "
        f"Лучший Macro-F1: {best_f1:.6f}"
    )

# ------------------------------------------------------------
# 5. Завершение
# ------------------------------------------------------------

total_hours = (
    time.time() - training_start_time
) / 3600

print("\n" + "=" * 60)
print("ЭКСПЕРИМЕНТ № 17 ЗАВЕРШЁН")
print("=" * 60)

print(f"Лучший Macro-F1: {best_f1:.6f}")
print(f"Время текущего запуска: {total_hours:.2f} ч")
print("Лучшая модель:", BEST_MODEL_PATH_RUBERT)
print("История:", HISTORY_PATH_RUBERT)
print("Checkpoint:", CHECKPOINT_PATH_RUBERT)


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. ВОССТАНОВЛЕНИЕ В НОВОМ COLAB
# ШАГ 1. ПРОВЕРКА GPU И CHECKPOINT
# ============================================================

import os
import torch
from google.colab import drive

# Подключаем Google Drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

# Путь к сохранённому checkpoint
CHECKPOINT_PATH = (
    "/content/drive/MyDrive/sentiment_research/"
    "experiment_17_rubert/latest_checkpoint.pt"
)

print("=" * 60)
print("ПРОВЕРКА НОВОГО ОКРУЖЕНИЯ")
print("=" * 60)

# Проверяем GPU
print("GPU доступен:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Проверяем checkpoint
if os.path.isfile(CHECKPOINT_PATH):
    print("Checkpoint найден: ДА")
    print(
        "Размер:",
        round(os.path.getsize(CHECKPOINT_PATH) / 1024**3, 2),
        "GB"
    )
else:
    print("Checkpoint найден: НЕТ")

print("\nПроверка переменных:")

variables = [
    "X_train_new",
    "y_train_new",
    "X_val_new",
    "y_val_new",
    "tokenizer_rubert",
    "train_dataset_rubert",
    "val_dataset_rubert",
    "model_rubert",
    "optimizer_rubert",
    "scheduler_rubert"
]

for name in variables:
    print(
        ("✅ " if name in globals() else "❌ ") + name
    )

print("\nОбучение не запускалось.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. ВОССТАНОВЛЕНИЕ ДАННЫХ
# ============================================================

import os
import re
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "merged_sentiment_dataset.csv.gz"
)

assert os.path.isfile(DATA_PATH), (
    "Не найден исходный датасет: " + DATA_PATH
)

print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ДАННЫХ RuBERT")
print("=" * 60)

# 1. Загружаем исходный датасет
df = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    low_memory=False
)

print("Всего строк:", len(df))

# 2. Исходная функция токенизации.
# Используем её только для восстановления
# того же отбора строк, что был ранее.
def tokenize_new(text):
    return re.findall(
        r"[а-яёa-z0-9]+",
        str(text).lower()
    )

# 3. Восстанавливаем разделения
train_new = df[
    df["split"] == "train"
].copy()

val_new = df[
    df["split"] == "validation"
].copy()

# 4. Убираем тексты без токенов,
# как в исходной предобработке.
train_new = train_new[
    train_new["text"].apply(
        lambda text: len(tokenize_new(text)) > 0
    )
].copy()

val_new = val_new[
    val_new["text"].apply(
        lambda text: len(tokenize_new(text)) > 0
    )
].copy()

# 5. Восстанавливаем тексты и метки
X_train_new = train_new["text"].astype(str).tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].astype(str).tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = [
    "Neutral",
    "Positive",
    "Negative"
]

# 6. Проверяем размеры
print("\nTrain:", len(X_train_new))
print("Validation:", len(X_val_new))

print(
    "Классы Train:",
    np.unique(y_train_new, return_counts=True)
)

print(
    "Классы Validation:",
    np.unique(y_val_new, return_counts=True)
)

assert len(X_train_new) == 203357, (
    "Размер Train отличается от исходного!"
)

assert len(X_val_new) == 20890, (
    "Размер Validation отличается от исходного!"
)

assert set(np.unique(y_train_new)) == {0, 1, 2}
assert set(np.unique(y_val_new)) == {0, 1, 2}

print("\n✅ Данные RuBERT восстановлены!")
print("Обучение не запускалось.")
print("Файлы на Google Drive не изменялись.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. ВОССТАНОВЛЕНИЕ ОКРУЖЕНИЯ
# ШАГ 4. TOKENIZER + DATASET + DATALOADER
# ============================================================

import sys
import subprocess
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

# 1. Проверяем библиотеку Transformers
try:
    import transformers
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers", "accelerate", "safetensors"
    ])
    import transformers

from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding
)

# 2. Настройки исходного эксперимента
MODEL_NAME_RUBERT = "DeepPavlov/rubert-base-cased"

MAX_LEN_RUBERT = 128
BATCH_SIZE_RUBERT = 8
GRAD_ACCUM_RUBERT = 4
SEED_RUBERT = 42

DEVICE_RUBERT = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE_RUBERT.type == "cuda", (
    "GPU недоступен. Включи T4 GPU в настройках Colab."
)

assert len(X_train_new) == 203357
assert len(X_val_new) == 20890

# 3. Загружаем исходный токенизатор
print("Загружаем токенизатор RuBERT...")

tokenizer_rubert = AutoTokenizer.from_pretrained(
    MODEL_NAME_RUBERT
)

# 4. Восстанавливаем Dataset
class RuBERTDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length):

        self.labels = np.asarray(
            labels, dtype=np.int64
        )

        print(f"Токенизация {len(texts):,} текстов...")

        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_length,
            padding=False,
            return_attention_mask=True
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": int(self.labels[idx])
        }

# 5. Восстанавливаем Train Dataset
print("\nПодготавливаем Train...")

train_dataset_rubert = RuBERTDataset(
    X_train_new,
    y_train_new,
    tokenizer_rubert,
    MAX_LEN_RUBERT
)

# 6. Восстанавливаем Validation Dataset
print("\nПодготавливаем Validation...")

val_dataset_rubert = RuBERTDataset(
    X_val_new,
    y_val_new,
    tokenizer_rubert,
    MAX_LEN_RUBERT
)

# 7. Динамический padding
collator_rubert = DataCollatorWithPadding(
    tokenizer=tokenizer_rubert,
    padding=True,
    return_tensors="pt"
)

# 8. DataLoader
train_loader_rubert = DataLoader(
    train_dataset_rubert,
    batch_size=BATCH_SIZE_RUBERT,
    shuffle=True,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

val_loader_rubert = DataLoader(
    val_dataset_rubert,
    batch_size=BATCH_SIZE_RUBERT,
    shuffle=False,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

# 9. Проверяем один батч
batch = next(iter(val_loader_rubert))

print("\n" + "=" * 60)
print("ПРОВЕРКА ВОССТАНОВЛЕНИЯ RuBERT")
print("=" * 60)

print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Train samples:", len(train_dataset_rubert))
print("Validation samples:", len(val_dataset_rubert))
print("Train batches:", len(train_loader_rubert))
print("Validation batches:", len(val_loader_rubert))

print("input_ids:", tuple(batch["input_ids"].shape))
print("attention_mask:", tuple(batch["attention_mask"].shape))
print("labels:", tuple(batch["labels"].shape))

assert len(train_loader_rubert) == 25420
assert len(val_loader_rubert) == 2612
assert batch["input_ids"].shape[1] <= 128

print("\n✅ Токенизатор и DataLoader восстановлены!")
print("Обучение не запускалось.")
print("Google Drive не изменялся.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 5. ВОССТАНОВЛЕНИЕ МОДЕЛИ И ОПТИМИЗАТОРА
# ============================================================

import os
import gc
import math
import torch

from torch.optim import AdamW
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

# 1. Пути
PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

CHECKPOINT_SOURCE_RUBERT = os.path.join(
    PROJECT_DIR,
    "experiment_17_rubert",
    "latest_checkpoint.pt"
)

assert os.path.isfile(CHECKPOINT_SOURCE_RUBERT), (
    "Checkpoint RuBERT не найден!"
)

# 2. Восстанавливаем исходную конфигурацию
NUM_LABELS_RUBERT = 3

ID2LABEL_RUBERT = {
    0: "Neutral",
    1: "Positive",
    2: "Negative"
}

LABEL2ID_RUBERT = {
    name: idx
    for idx, name in ID2LABEL_RUBERT.items()
}

EPOCHS_RUBERT = 3
LR_RUBERT = 2e-5
WEIGHT_DECAY_RUBERT = 0.01
WARMUP_RATIO_RUBERT = 0.1
MAX_GRAD_NORM_RUBERT = 1.0

STEPS_PER_EPOCH_RUBERT = math.ceil(
    len(train_loader_rubert) / GRAD_ACCUM_RUBERT
)

TOTAL_STEPS_RUBERT = (
    STEPS_PER_EPOCH_RUBERT * EPOCHS_RUBERT
)

WARMUP_STEPS_RUBERT = int(
    TOTAL_STEPS_RUBERT * WARMUP_RATIO_RUBERT
)

# 3. Загружаем checkpoint в оперативную память
print("Читаем checkpoint с Google Drive...")

checkpoint_rubert = torch.load(
    CHECKPOINT_SOURCE_RUBERT,
    map_location="cpu",
    weights_only=False
)

# 4. Создаём архитектуру RuBERT
# Веса затем полностью заменим на сохранённые.
print("Создаём архитектуру RuBERT...")

model_rubert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_RUBERT,
    num_labels=NUM_LABELS_RUBERT,
    id2label=ID2LABEL_RUBERT,
    label2id=LABEL2ID_RUBERT
)

# 5. Восстанавливаем веса
model_rubert.load_state_dict(
    checkpoint_rubert["model_state_dict"],
    strict=True
)

model_rubert = model_rubert.to(DEVICE_RUBERT)

# 6. Восстанавливаем optimizer
optimizer_rubert = AdamW(
    model_rubert.parameters(),
    lr=LR_RUBERT,
    weight_decay=WEIGHT_DECAY_RUBERT
)

optimizer_rubert.load_state_dict(
    checkpoint_rubert["optimizer_state_dict"]
)

# 7. Восстанавливаем scheduler
scheduler_rubert = get_linear_schedule_with_warmup(
    optimizer_rubert,
    num_warmup_steps=WARMUP_STEPS_RUBERT,
    num_training_steps=TOTAL_STEPS_RUBERT
)

scheduler_rubert.load_state_dict(
    checkpoint_rubert["scheduler_state_dict"]
)

# 8. Восстанавливаем счётчики
start_epoch_rubert = checkpoint_rubert["epoch"]
start_batch_idx_rubert = checkpoint_rubert["next_batch_idx"]
global_step_rubert = checkpoint_rubert["global_step"]
best_f1_rubert = checkpoint_rubert["best_f1"]

resume_loss_sum_rubert = checkpoint_rubert.get(
    "epoch_loss_sum", 0.0
)

resume_loss_count_rubert = checkpoint_rubert.get(
    "epoch_loss_count", 0
)

# Сохраняем состояния RNG для будущего продолжения
rng_state_rubert = {
    "torch": checkpoint_rubert.get("torch_rng_state"),
    "numpy": checkpoint_rubert.get("numpy_rng_state"),
    "python": checkpoint_rubert.get("python_rng_state"),
    "cuda": checkpoint_rubert.get("cuda_rng_state")
}

# 9. Освобождаем оперативную память
del checkpoint_rubert
gc.collect()
torch.cuda.empty_cache()

# 10. Проверяем результат
print("\n" + "=" * 60)
print("ВОССТАНОВЛЕНИЕ RuBERT ЗАВЕРШЕНО")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Эпоха:", start_epoch_rubert + 1)
print("Следующий батч:", start_batch_idx_rubert)
print("Optimizer step:", global_step_rubert)
print("Лучший Macro-F1:", best_f1_rubert)

print(
    "Learning rate:",
    optimizer_rubert.param_groups[0]["lr"]
)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

assert global_step_rubert == 5500
assert start_epoch_rubert == 0
assert start_batch_idx_rubert == 22000

print("\n✅ RuBERT восстановлен с шага 5500!")
print("Обучение не запускалось.")
print("Checkpoint на Google Drive не изменён.")


In [ ]:

# ============================================================
# ПРОВЕРКА RAM ПОСЛЕ ПЕРЕЗАПУСКА COLAB
# ============================================================

import os
import torch
import psutil

ram = psutil.virtual_memory()

print("=" * 60)
print("ПРОВЕРКА ПАМЯТИ COLAB")
print("=" * 60)

print(
    "Всего RAM:",
    round(ram.total / 1024**3, 2),
    "GB"
)

print(
    "Свободно RAM:",
    round(ram.available / 1024**3, 2),
    "GB"
)

print(
    "Используется RAM:",
    round(ram.used / 1024**3, 2),
    "GB"
)

print("GPU доступен:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nПроверка завершена.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 6. СОЗДАНИЕ АРХИТЕКТУРЫ БЕЗ ВЕСОВ
# ============================================================

import gc
import torch
import psutil

from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification
)

# 1. Исходная конфигурация
MODEL_NAME_RUBERT = "DeepPavlov/rubert-base-cased"

NUM_LABELS_RUBERT = 3

ID2LABEL_RUBERT = {
    0: "Neutral",
    1: "Positive",
    2: "Negative"
}

LABEL2ID_RUBERT = {
    name: idx
    for idx, name in ID2LABEL_RUBERT.items()
}

DEVICE_RUBERT = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE_RUBERT.type == "cuda", (
    "GPU недоступен!"
)

# 2. Освобождаем память
gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("ЭКОНОМНОЕ СОЗДАНИЕ RuBERT")
print("=" * 60)

print(
    "RAM до создания:",
    round(
        psutil.virtual_memory().available / 1024**3,
        2
    ),
    "GB"
)

# 3. Загружаем только конфигурацию
print("\nЗагружаем конфигурацию модели...")

config_rubert = AutoConfig.from_pretrained(
    MODEL_NAME_RUBERT,
    num_labels=NUM_LABELS_RUBERT,
    id2label=ID2LABEL_RUBERT,
    label2id=LABEL2ID_RUBERT
)

# 4. Создаём архитектуру без предобученных весов
print("Создаём архитектуру без скачивания весов...")

model_rubert = (
    AutoModelForSequenceClassification.from_config(
        config_rubert
    )
)

# Пока оставляем модель на CPU.
# Перенос на GPU выполним после загрузки checkpoint.

# 5. Проверяем модель
total_params = sum(
    p.numel() for p in model_rubert.parameters()
)

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ")
print("=" * 60)

print("Модель:", MODEL_NAME_RUBERT)
print("Параметров:", f"{total_params:,}")
print("Классов:", model_rubert.config.num_labels)

print(
    "RAM свободно:",
    round(
        psutil.virtual_memory().available / 1024**3,
        2
    ),
    "GB"
)

print(
    "GPU memory allocated:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2
    ),
    "GB"
)

assert model_rubert.config.num_labels == 3

print("\n✅ Архитектура RuBERT создана!")
print("Обучение не запускалось.")
print("Checkpoint не изменялся.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 7. ЭКОНОМНАЯ ЗАГРУЗКА ВЕСОВ ИЗ CHECKPOINT
# ============================================================

import os
import gc
import torch
import psutil

CHECKPOINT_SOURCE_RUBERT = (
    "/content/drive/MyDrive/sentiment_research/"
    "experiment_17_rubert/latest_checkpoint.pt"
)

assert os.path.isfile(CHECKPOINT_SOURCE_RUBERT), (
    "Checkpoint не найден!"
)

assert "model_rubert" in globals(), (
    "Архитектура RuBERT не создана!"
)

print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ВЕСОВ RuBERT")
print("=" * 60)

print(
    "RAM до загрузки:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

# 1. Загружаем checkpoint на CPU
# weights_only=False допустим только для нашего
# собственного доверенного checkpoint.
print("\nЧитаем checkpoint...")

checkpoint_rubert = torch.load(
    CHECKPOINT_SOURCE_RUBERT,
    map_location="cpu",
    weights_only=False,
    mmap=True
)

# 2. Проверяем сохранённый прогресс
print("\nСохранённый шаг:",
      checkpoint_rubert["global_step"])

print("Эпоха:",
      checkpoint_rubert["epoch"] + 1)

assert checkpoint_rubert["global_step"] == 5500

# 3. Загружаем веса в существующую модель
print("\nВосстанавливаем веса модели...")

model_rubert.load_state_dict(
    checkpoint_rubert["model_state_dict"],
    strict=True,
    assign=True
)

# 4. Сохраняем метаданные
restored_step_rubert = checkpoint_rubert["global_step"]
restored_epoch_rubert = checkpoint_rubert["epoch"]
restored_batch_rubert = checkpoint_rubert["next_batch_idx"]

# 5. Освобождаем память от checkpoint
# Оптимизатор восстановим отдельно позже.
del checkpoint_rubert

gc.collect()
torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ")
print("=" * 60)

print("Восстановленный шаг:", restored_step_rubert)
print("Эпоха:", restored_epoch_rubert + 1)
print("Следующий батч:", restored_batch_rubert)

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\n✅ Веса RuBERT восстановлены!")
print("Обучение не запускалось.")
print("Checkpoint на Google Drive не изменён.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 8. ВОССТАНОВЛЕНИЕ ADAMW И SCHEDULER
# ============================================================

import gc
import math
import random
import numpy as np
import torch
import psutil

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# 1. Конфигурация исходного эксперимента
SEED_RUBERT = 42
EPOCHS_RUBERT = 3

BATCH_SIZE_RUBERT = 8
GRAD_ACCUM_RUBERT = 4

LR_RUBERT = 2e-5
WEIGHT_DECAY_RUBERT = 0.01
WARMUP_RATIO_RUBERT = 0.1
MAX_GRAD_NORM_RUBERT = 1.0

TRAIN_SAMPLES_RUBERT = 203357

TRAIN_BATCHES_RUBERT = math.ceil(
    TRAIN_SAMPLES_RUBERT / BATCH_SIZE_RUBERT
)

STEPS_PER_EPOCH_RUBERT = math.ceil(
    TRAIN_BATCHES_RUBERT / GRAD_ACCUM_RUBERT
)

TOTAL_STEPS_RUBERT = (
    STEPS_PER_EPOCH_RUBERT * EPOCHS_RUBERT
)

WARMUP_STEPS_RUBERT = int(
    TOTAL_STEPS_RUBERT * WARMUP_RATIO_RUBERT
)

assert restored_step_rubert == 5500
assert restored_epoch_rubert == 0
assert restored_batch_rubert == 22000

# 2. Создаём оптимизатор ПОСЛЕ assign=True
optimizer_rubert = AdamW(
    model_rubert.parameters(),
    lr=LR_RUBERT,
    weight_decay=WEIGHT_DECAY_RUBERT
)

# 3. Создаём scheduler
scheduler_rubert = get_linear_schedule_with_warmup(
    optimizer_rubert,
    num_warmup_steps=WARMUP_STEPS_RUBERT,
    num_training_steps=TOTAL_STEPS_RUBERT
)

print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ОПТИМИЗАТОРА RuBERT")
print("=" * 60)

print(
    "RAM до загрузки:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

# 4. Читаем checkpoint экономно
checkpoint_optimizer = torch.load(
    CHECKPOINT_SOURCE_RUBERT,
    map_location="cpu",
    weights_only=False,
    mmap=True
)

# 5. Восстанавливаем состояние AdamW
optimizer_rubert.load_state_dict(
    checkpoint_optimizer["optimizer_state_dict"]
)

# 6. Восстанавливаем scheduler
scheduler_rubert.load_state_dict(
    checkpoint_optimizer["scheduler_state_dict"]
)

# 7. Восстанавливаем дополнительные данные
start_epoch_rubert = checkpoint_optimizer["epoch"]
start_batch_idx_rubert = checkpoint_optimizer["next_batch_idx"]
global_step_rubert = checkpoint_optimizer["global_step"]
best_f1_rubert = checkpoint_optimizer["best_f1"]

resume_loss_sum_rubert = checkpoint_optimizer.get(
    "epoch_loss_sum", 0.0
)

resume_loss_count_rubert = checkpoint_optimizer.get(
    "epoch_loss_count", 0
)

rng_state_rubert = {
    "torch": checkpoint_optimizer.get("torch_rng_state"),
    "numpy": checkpoint_optimizer.get("numpy_rng_state"),
    "python": checkpoint_optimizer.get("python_rng_state"),
    "cuda": checkpoint_optimizer.get("cuda_rng_state")
}

# 8. Освобождаем временный checkpoint
del checkpoint_optimizer
gc.collect()

# 9. Проверяем результат
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ")
print("=" * 60)

print("Optimizer step:", global_step_rubert)
print("Эпоха:", start_epoch_rubert + 1)
print("Следующий батч:", start_batch_idx_rubert)

print(
    "Learning rate:",
    optimizer_rubert.param_groups[0]["lr"]
)

print(
    "Состояний параметров AdamW:",
    len(optimizer_rubert.state)
)

print(
    "Scheduler last_epoch:",
    scheduler_rubert.last_epoch
)

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

assert global_step_rubert == 5500
assert len(optimizer_rubert.state) > 0

print("\n✅ Оптимизатор и scheduler восстановлены!")
print("Обучение не запускалось.")
print("Google Drive не изменялся.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 8. ВОССТАНОВЛЕНИЕ ADAMW И SCHEDULER
# ============================================================

import gc
import math
import random
import numpy as np
import torch
import psutil

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# 1. Конфигурация исходного эксперимента
SEED_RUBERT = 42
EPOCHS_RUBERT = 3

BATCH_SIZE_RUBERT = 8
GRAD_ACCUM_RUBERT = 4

LR_RUBERT = 2e-5
WEIGHT_DECAY_RUBERT = 0.01
WARMUP_RATIO_RUBERT = 0.1
MAX_GRAD_NORM_RUBERT = 1.0

TRAIN_SAMPLES_RUBERT = 203357

TRAIN_BATCHES_RUBERT = math.ceil(
    TRAIN_SAMPLES_RUBERT / BATCH_SIZE_RUBERT
)

STEPS_PER_EPOCH_RUBERT = math.ceil(
    TRAIN_BATCHES_RUBERT / GRAD_ACCUM_RUBERT
)

TOTAL_STEPS_RUBERT = (
    STEPS_PER_EPOCH_RUBERT * EPOCHS_RUBERT
)

WARMUP_STEPS_RUBERT = int(
    TOTAL_STEPS_RUBERT * WARMUP_RATIO_RUBERT
)

assert restored_step_rubert == 5500
assert restored_epoch_rubert == 0
assert restored_batch_rubert == 22000

# 2. Создаём оптимизатор ПОСЛЕ assign=True
optimizer_rubert = AdamW(
    model_rubert.parameters(),
    lr=LR_RUBERT,
    weight_decay=WEIGHT_DECAY_RUBERT
)

# 3. Создаём scheduler
scheduler_rubert = get_linear_schedule_with_warmup(
    optimizer_rubert,
    num_warmup_steps=WARMUP_STEPS_RUBERT,
    num_training_steps=TOTAL_STEPS_RUBERT
)

print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ОПТИМИЗАТОРА RuBERT")
print("=" * 60)

print(
    "RAM до загрузки:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

# 4. Читаем checkpoint экономно
checkpoint_optimizer = torch.load(
    CHECKPOINT_SOURCE_RUBERT,
    map_location="cpu",
    weights_only=False,
    mmap=True
)

# 5. Восстанавливаем состояние AdamW
optimizer_rubert.load_state_dict(
    checkpoint_optimizer["optimizer_state_dict"]
)

# 6. Восстанавливаем scheduler
scheduler_rubert.load_state_dict(
    checkpoint_optimizer["scheduler_state_dict"]
)

# 7. Восстанавливаем дополнительные данные
start_epoch_rubert = checkpoint_optimizer["epoch"]
start_batch_idx_rubert = checkpoint_optimizer["next_batch_idx"]
global_step_rubert = checkpoint_optimizer["global_step"]
best_f1_rubert = checkpoint_optimizer["best_f1"]

resume_loss_sum_rubert = checkpoint_optimizer.get(
    "epoch_loss_sum", 0.0
)

resume_loss_count_rubert = checkpoint_optimizer.get(
    "epoch_loss_count", 0
)

rng_state_rubert = {
    "torch": checkpoint_optimizer.get("torch_rng_state"),
    "numpy": checkpoint_optimizer.get("numpy_rng_state"),
    "python": checkpoint_optimizer.get("python_rng_state"),
    "cuda": checkpoint_optimizer.get("cuda_rng_state")
}

# 8. Освобождаем временный checkpoint
del checkpoint_optimizer
gc.collect()

# 9. Проверяем результат
print("\n" + "=" * 60)
print("РЕЗУЛЬТАТ")
print("=" * 60)

print("Optimizer step:", global_step_rubert)
print("Эпоха:", start_epoch_rubert + 1)
print("Следующий батч:", start_batch_idx_rubert)

print(
    "Learning rate:",
    optimizer_rubert.param_groups[0]["lr"]
)

print(
    "Состояний параметров AdamW:",
    len(optimizer_rubert.state)
)

print(
    "Scheduler last_epoch:",
    scheduler_rubert.last_epoch
)

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

assert global_step_rubert == 5500
assert len(optimizer_rubert.state) > 0

print("\n✅ Оптимизатор и scheduler восстановлены!")
print("Обучение не запускалось.")
print("Google Drive не изменялся.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 9. ЭКОНОМНОЕ ВОССТАНОВЛЕНИЕ ДАННЫХ
# ============================================================

import os
import re
import gc
import psutil
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# ------------------------------------------------------------
# 1. Загружаем исходные данные
# ------------------------------------------------------------

DATA_PATH = (
    "/content/drive/MyDrive/sentiment_research/"
    "merged_sentiment_dataset.csv.gz"
)

print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ДАННЫХ RuBERT")
print("=" * 60)

print(
    "Свободно RAM до загрузки:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

df_rubert = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    low_memory=False
)

def tokenize_new(text):
    return re.findall(
        r"[а-яёa-z0-9]+",
        str(text).lower()
    )

train_new = df_rubert[
    df_rubert["split"] == "train"
].copy()

val_new = df_rubert[
    df_rubert["split"] == "validation"
].copy()

train_new = train_new[
    train_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

val_new = val_new[
    val_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

X_train_new = train_new["text"].astype(str).tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].astype(str).tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

assert len(X_train_new) == 203357
assert len(X_val_new) == 20890

# Удаляем лишние DataFrame
del df_rubert, train_new, val_new
gc.collect()

# ------------------------------------------------------------
# 2. Загружаем токенизатор
# ------------------------------------------------------------

print("\nЗагружаем токенизатор...")

tokenizer_rubert = AutoTokenizer.from_pretrained(
    "DeepPavlov/rubert-base-cased"
)

# ------------------------------------------------------------
# 3. Dataset с токенизацией на лету
# ------------------------------------------------------------

class RuBERTLazyDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "labels": int(self.labels[idx])
        }


train_dataset_rubert = RuBERTLazyDataset(
    X_train_new,
    y_train_new
)

val_dataset_rubert = RuBERTLazyDataset(
    X_val_new,
    y_val_new
)

# ------------------------------------------------------------
# 4. Токенизация батча
# ------------------------------------------------------------

def collator_rubert(batch):

    texts = [item["text"] for item in batch]

    labels = torch.tensor(
        [item["labels"] for item in batch],
        dtype=torch.long
    )

    encoded = tokenizer_rubert(
        texts,
        truncation=True,
        max_length=128,
        padding=True,
        return_tensors="pt"
    )

    encoded["labels"] = labels

    return encoded


# ------------------------------------------------------------
# 5. DataLoader
# ------------------------------------------------------------

train_loader_rubert = DataLoader(
    train_dataset_rubert,
    batch_size=8,
    shuffle=True,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

val_loader_rubert = DataLoader(
    val_dataset_rubert,
    batch_size=8,
    shuffle=False,
    collate_fn=collator_rubert,
    num_workers=0,
    pin_memory=True
)

# ------------------------------------------------------------
# 6. Проверка
# ------------------------------------------------------------

batch = next(iter(val_loader_rubert))

print("\n" + "=" * 60)
print("ПРОВЕРКА")
print("=" * 60)

print("Train:", len(train_dataset_rubert))
print("Validation:", len(val_dataset_rubert))

print("Train batches:", len(train_loader_rubert))
print("Validation batches:", len(val_loader_rubert))

print("input_ids:", tuple(batch["input_ids"].shape))
print("attention_mask:", tuple(batch["attention_mask"].shape))
print("labels:", tuple(batch["labels"].shape))

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

assert len(train_loader_rubert) == 25420
assert len(val_loader_rubert) == 2612

print("\n✅ Данные RuBERT восстановлены!")
print("Обучение не запускалось.")
print("Google Drive не изменялся.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 10. ПОДГОТОВКА ЛОКАЛЬНОГО ХРАНИЛИЩА
# ============================================================

import os
import shutil
import psutil

# 1. Новая папка для checkpoint
EXP17_LOCAL_DIR = "/content/rubert_checkpoints"

os.makedirs(EXP17_LOCAL_DIR, exist_ok=True)

# 2. Пути для новых файлов
CHECKPOINT_PATH_RUBERT = os.path.join(
    EXP17_LOCAL_DIR,
    "latest_checkpoint.pt"
)

BEST_MODEL_PATH_RUBERT = os.path.join(
    EXP17_LOCAL_DIR,
    "best_model.pt"
)

HISTORY_PATH_RUBERT = os.path.join(
    EXP17_LOCAL_DIR,
    "training_history.csv"
)

# 3. Проверяем свободное место
disk = shutil.disk_usage(EXP17_LOCAL_DIR)

free_gb = disk.free / 1024**3
total_gb = disk.total / 1024**3

print("=" * 60)
print("ЛОКАЛЬНОЕ ХРАНИЛИЩЕ RuBERT")
print("=" * 60)

print("Папка:", EXP17_LOCAL_DIR)
print(f"Всего места: {total_gb:.2f} GB")
print(f"Свободно: {free_gb:.2f} GB")

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

print("\nПути сохранения:")
print("Checkpoint:", CHECKPOINT_PATH_RUBERT)
print("Лучшая модель:", BEST_MODEL_PATH_RUBERT)
print("История:", HISTORY_PATH_RUBERT)

# 4. Проверяем возможность сохранения
if free_gb < 6:
    raise RuntimeError(
        "Недостаточно свободного места "
        "для безопасного сохранения checkpoint."
    )

print("\n✅ Локальное хранилище готово!")
print("Google Drive не используется для новых сохранений.")
print("Обучение не запускалось.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 11. ПРОВЕРКА ЛОКАЛЬНОГО CHECKPOINT
# ============================================================

import os
import gc
import random
import torch
import numpy as np
import psutil

print("=" * 60)
print("ПРОВЕРКА ЛОКАЛЬНОГО СОХРАНЕНИЯ")
print("=" * 60)

# 1. Проверяем восстановленные объекты
required = [
    "model_rubert",
    "optimizer_rubert",
    "scheduler_rubert",
    "global_step_rubert",
    "start_epoch_rubert",
    "start_batch_idx_rubert",
    "CHECKPOINT_PATH_RUBERT"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Отсутствуют переменные: " + ", ".join(missing)
    )

assert global_step_rubert == 5500

# 2. Сохраняем состояние.
# Для восстановления исходного порядка случайных операций
# используем RNG из checkpoint, если он доступен.
rng = globals().get("rng_state_rubert", {})

checkpoint_data = {
    "epoch": start_epoch_rubert,
    "next_batch_idx": start_batch_idx_rubert,
    "global_step": global_step_rubert,
    "best_f1": best_f1_rubert,
    "epoch_loss_sum": resume_loss_sum_rubert,
    "epoch_loss_count": resume_loss_count_rubert,
    "model_state_dict": model_rubert.state_dict(),
    "optimizer_state_dict": optimizer_rubert.state_dict(),
    "scheduler_state_dict": scheduler_rubert.state_dict(),
    "torch_rng_state": rng.get("torch"),
    "numpy_rng_state": rng.get("numpy"),
    "python_rng_state": rng.get("python"),
    "cuda_rng_state": rng.get("cuda")
}

temp_path = CHECKPOINT_PATH_RUBERT + ".tmp"

print("Сохраняем локальный checkpoint...")

torch.save(checkpoint_data, temp_path)
os.replace(temp_path, CHECKPOINT_PATH_RUBERT)

del checkpoint_data
gc.collect()

# 3. Проверяем размер
size_gb = os.path.getsize(
    CHECKPOINT_PATH_RUBERT
) / 1024**3

print(f"Размер файла: {size_gb:.2f} GB")

# 4. Проверяем чтение без повторного копирования весов
print("Проверяем содержимое...")

test_checkpoint = torch.load(
    CHECKPOINT_PATH_RUBERT,
    map_location="cpu",
    weights_only=False,
    mmap=True
)

assert test_checkpoint["global_step"] == 5500
assert test_checkpoint["next_batch_idx"] == 22000

assert "model_state_dict" in test_checkpoint
assert "optimizer_state_dict" in test_checkpoint
assert "scheduler_state_dict" in test_checkpoint

print("Optimizer step:", test_checkpoint["global_step"])
print("Следующий батч:", test_checkpoint["next_batch_idx"])
print("Веса модели: OK")
print("Оптимизатор: OK")
print("Scheduler: OK")

del test_checkpoint
gc.collect()

print(
    "Свободно RAM:",
    round(psutil.virtual_memory().available / 1024**3, 2),
    "GB"
)

print("\n✅ Локальный checkpoint сохранён и прочитан!")
print("Google Drive не изменялся.")
print("Обучение не запускалось.")


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ШАГ 12. СКАЧИВАНИЕ CHECKPOINT НА MAC
# ============================================================

import os
from google.colab import files

assert os.path.isfile(CHECKPOINT_PATH_RUBERT), (
    "Локальный checkpoint не найден!"
)

size_gb = os.path.getsize(
    CHECKPOINT_PATH_RUBERT
) / 1024**3

print("=" * 60)
print("РЕЗЕРВНОЕ КОПИРОВАНИЕ RuBERT")
print("=" * 60)

print("Файл:", CHECKPOINT_PATH_RUBERT)
print(f"Размер: {size_gb:.2f} GB")
print("Шаг обучения: 5500")

print("\nЗапускаем скачивание...")
print("Дождись полного завершения загрузки в браузере.")

files.download(CHECKPOINT_PATH_RUBERT)


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. RuBERT
# ПРОДОЛЖЕНИЕ ОБУЧЕНИЯ С ШАГА 5500
# ============================================================

import os
import gc
import math
import random
import time
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

# 1. Проверки

required = [
    "model_rubert", "optimizer_rubert", "scheduler_rubert",
    "train_dataset_rubert", "val_loader_rubert",
    "collator_rubert", "CHECKPOINT_PATH_RUBERT",
    "global_step_rubert", "start_epoch_rubert",
    "start_batch_idx_rubert", "rng_state_rubert"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError("Не найдены: " + ", ".join(missing))

assert global_step_rubert == 5500, (
    "Ожидался старт с шага 5500. "
    "Не запускай эту ячейку повторно после начала обучения."
)

assert start_batch_idx_rubert == 22000

# 2. Настройки

DEVICE_RUBERT = torch.device("cuda")
SAVE_EVERY_RUBERT = 500

os.makedirs(EXP17_LOCAL_DIR, exist_ok=True)

# 3. Переносим модель и состояния AdamW на GPU

print("Переносим RuBERT на GPU...")
model_rubert.to(DEVICE_RUBERT)

for state in optimizer_rubert.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(DEVICE_RUBERT)

torch.cuda.empty_cache()
gc.collect()

print("GPU:", torch.cuda.get_device_name(0))
print("Продолжаем с шага:", global_step_rubert)

# 4. Восстанавливаем RNG

if rng_state_rubert.get("torch") is not None:
    torch.set_rng_state(rng_state_rubert["torch"])

if rng_state_rubert.get("numpy") is not None:
    np.random.set_state(rng_state_rubert["numpy"])

if rng_state_rubert.get("python") is not None:
    random.setstate(rng_state_rubert["python"])

if rng_state_rubert.get("cuda") is not None:
    torch.cuda.set_rng_state_all(rng_state_rubert["cuda"])

# 5. DataLoader с исходным перемешиванием

def make_train_loader_rubert(epoch):
    generator = torch.Generator()
    generator.manual_seed(SEED_RUBERT + epoch)

    return DataLoader(
        train_dataset_rubert,
        batch_size=BATCH_SIZE_RUBERT,
        shuffle=True,
        generator=generator,
        collate_fn=collator_rubert,
        num_workers=0,
        pin_memory=True
    )

# 6. Функция сохранения

def save_checkpoint_rubert(
    epoch, next_batch_idx, global_step,
    epoch_loss_sum, epoch_loss_count, best_f1
):
    state = {
        "epoch": epoch,
        "next_batch_idx": next_batch_idx,
        "global_step": global_step,
        "best_f1": best_f1,
        "epoch_loss_sum": epoch_loss_sum,
        "epoch_loss_count": epoch_loss_count,
        "model_state_dict": model_rubert.state_dict(),
        "optimizer_state_dict": optimizer_rubert.state_dict(),
        "scheduler_state_dict": scheduler_rubert.state_dict(),
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": torch.cuda.get_rng_state_all()
    }

    temp_path = CHECKPOINT_PATH_RUBERT + ".tmp"
    torch.save(state, temp_path)
    os.replace(temp_path, CHECKPOINT_PATH_RUBERT)

    del state
    gc.collect()

    print(f"\nCheckpoint сохранён: шаг {global_step}")


# 7. Валидация

@torch.no_grad()
def evaluate_rubert():
    model_rubert.eval()

    predictions = []
    targets = []

    for batch in tqdm(
        val_loader_rubert,
        desc="Validation",
        leave=False
    ):
        batch = {
            k: v.to(DEVICE_RUBERT)
            for k, v in batch.items()
        }

        outputs = model_rubert(**batch)

        preds = outputs.logits.argmax(dim=1)

        predictions.extend(preds.cpu().tolist())
        targets.extend(batch["labels"].cpu().tolist())

    return {
        "macro_f1": f1_score(
            targets, predictions, average="macro"
        ),
        "accuracy": accuracy_score(
            targets, predictions
        )
    }


# 8. Обучение

history_rubert = []

print("\n" + "=" * 60)
print("RuBERT — ПРОДОЛЖЕНИЕ ОБУЧЕНИЯ")
print("=" * 60)

print("Начальный шаг:", global_step_rubert)
print("Всего шагов:", TOTAL_STEPS_RUBERT)

for epoch in range(start_epoch_rubert, EPOCHS_RUBERT):

    model_rubert.train()

    train_loader = make_train_loader_rubert(epoch)

    first_batch = (
        start_batch_idx_rubert
        if epoch == start_epoch_rubert
        else 0
    )

    epoch_loss_sum = (
        resume_loss_sum_rubert
        if epoch == start_epoch_rubert
        else 0.0
    )

    epoch_loss_count = (
        resume_loss_count_rubert
        if epoch == start_epoch_rubert
        else 0
    )

    optimizer_rubert.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS_RUBERT}"
    )

    for batch_idx, batch in progress:

        if batch_idx < first_batch:
            continue

        batch = {
            k: v.to(DEVICE_RUBERT)
            for k, v in batch.items()
        }

        outputs = model_rubert(**batch)

        group_start = (
            batch_idx // GRAD_ACCUM_RUBERT
        ) * GRAD_ACCUM_RUBERT

        group_size = min(
            GRAD_ACCUM_RUBERT,
            len(train_loader) - group_start
        )

        loss = outputs.loss / group_size
        loss.backward()

        epoch_loss_sum += outputs.loss.item()
        epoch_loss_count += 1

        is_update = (
            (batch_idx + 1) % GRAD_ACCUM_RUBERT == 0
            or batch_idx + 1 == len(train_loader)
        )

        if is_update:

            torch.nn.utils.clip_grad_norm_(
                model_rubert.parameters(),
                MAX_GRAD_NORM_RUBERT
            )

            optimizer_rubert.step()
            scheduler_rubert.step()
            optimizer_rubert.zero_grad(set_to_none=True)

            global_step_rubert += 1

            progress.set_postfix(
                step=global_step_rubert,
                loss=round(outputs.loss.item(), 4)
            )

            if global_step_rubert % SAVE_EVERY_RUBERT == 0:
                save_checkpoint_rubert(
                    epoch,
                    batch_idx + 1,
                    global_step_rubert,
                    epoch_loss_sum,
                    epoch_loss_count,
                    best_f1_rubert
                )

    # 9. Валидация после эпохи

    print(f"\nВалидация эпохи {epoch + 1}...")

    metrics = evaluate_rubert()

    avg_loss = epoch_loss_sum / max(epoch_loss_count, 1)

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Macro F1: {metrics['macro_f1']:.4f} | "
        f"Accuracy: {metrics['accuracy']:.4f}"
    )

    if metrics["macro_f1"] > best_f1_rubert:
        best_f1_rubert = metrics["macro_f1"]

        torch.save(
            model_rubert.state_dict(),
            BEST_MODEL_PATH_RUBERT
        )

        print("Новая лучшая модель сохранена!")

    history_rubert.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "macro_f1": metrics["macro_f1"],
        "accuracy": metrics["accuracy"]
    })

    pd.DataFrame(history_rubert).to_csv(
        HISTORY_PATH_RUBERT,
        index=False
    )

    # Сохраняем состояние после завершения эпохи.
    save_checkpoint_rubert(
        epoch + 1,
        0,
        global_step_rubert,
        0.0,
        0,
        best_f1_rubert
    )

    model_rubert.train()

    # После первой эпохи продолжаем с начала следующей
    start_batch_idx_rubert = 0

print("\n" + "=" * 60)
print("ОБУЧЕНИЕ RuBERT ЗАВЕРШЕНО")
print("=" * 60)

print("Последний шаг:", global_step_rubert)
print("Лучший Macro F1:", round(best_f1_rubert, 4))
print("Checkpoint:", CHECKPOINT_PATH_RUBERT)
print("Лучшая модель:", BEST_MODEL_PATH_RUBERT)


In [ ]:

import os
from google.colab import files

path = "/content/rubert_checkpoints/best_model.pt"

assert os.path.isfile(path), "Лучшая модель не найдена!"

print("Лучшая модель RuBERT")
print("Macro F1: 0.8013")
print("Размер:", round(os.path.getsize(path) / 1024**2, 1), "MB")

files.download(path)


In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 17. ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files

results_rubert = pd.DataFrame({
    "experiment": [17, 17, 17],
    "model": ["RuBERT"] * 3,
    "epoch": [1, 2, 3],
    "train_loss": [0.5062, 0.3540, 0.2588],
    "macro_f1": [0.7788, 0.8013, 0.7993],
    "accuracy": [0.7891, 0.8101, 0.8080]
})

best_idx = results_rubert["macro_f1"].idxmax()
best_result = results_rubert.loc[best_idx]

print("=" * 60)
print("ЭКСПЕРИМЕНТ 17 — RuBERT")
print("=" * 60)

display(results_rubert)

print("\nЛучшая эпоха:", int(best_result["epoch"]))
print("Лучший Macro F1:", best_result["macro_f1"])
print("Accuracy:", best_result["accuracy"])

# Графики динамики качества
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(
    results_rubert["epoch"],
    results_rubert["macro_f1"],
    marker="o",
    label="Macro F1"
)

ax.plot(
    results_rubert["epoch"],
    results_rubert["accuracy"],
    marker="s",
    label="Accuracy"
)

ax.set_title("RuBERT: качество по эпохам")
ax.set_xlabel("Эпоха")
ax.set_ylabel("Значение метрики")
ax.set_xticks([1, 2, 3])
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# Сохраняем таблицу локально
output_path = "/content/rubert_experiment_17_results.csv"

results_rubert.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nТаблица сохранена:", output_path)

# Скачиваем небольшой CSV на компьютер
files.download(output_path)
